# 🫀 퀘스트 46 · Q4-N — **범위(AAMI)·운영점·P 벡터축·순위손실**

| | **MedKOS / `notebooks/quest46_q4n_scope_rank_vector.ipynb`** |
|---|---|
| 퀘스트 | `ailab-2026-0046` — 축을 더 얹기 전에 **자·범위·목적함수**를 맞춘다 |
| 부모 런 | `quest46_q4m_pont_cluster_boost`(`20260805T1436`) |
| 예상 소요 | **45–55분**(CPU) + 딥러닝 3변형 **10–15분**(GPU 있을 때만) |

## ★★★ Q4-M — 자는 완전히 섰는데 **새 축 둘이 다 안 얹혔다**

```
팔       차원  k-스윕   달성률@300  AUROC   PR-AUC
base       9  0.6791   0.8074    0.9420  0.5796
morph     17  0.8361   0.9018    0.9529  0.7236   ← 세 런 연속 재현
pont      23  0.8293   0.8933    0.9493  0.7157
pshuf     23  0.8360   0.8998    0.9532  0.7230
clus      21  0.8307   0.8985    0.9513  0.7118
cshuf     21  0.8356   0.9019    0.9533  0.7235
comb      27  0.8231   0.8860    0.9463  0.7016

morph − base    +0.1570 [+0.1005, +0.2168]  ✅  (Q4-L 과 **소수점까지 동일**)
pont − pshuf    −0.0067 [−0.0169, +0.0021]  ⚠️
clus − cshuf    −0.0049 [−0.0157, +0.0059]  ⚠️
comb − morph    −0.0130 [−0.0282, +0.0022]  ⚠️
```

**잘된 것** — 자 검증 전부 통과(QRS **77.8ms** · PR **131.9ms** · P 폭 **75.0ms** · 역위 P
**20.2%** · 검출률 N 95.4% > V 87.7% · 리드 분리 64.1% · **물린 레코드 진단이 작동**),
중복 감사 v2 회고에서 `tp_over_rr` **|ρ| 1.0000**, 형태 축 **세 번째 재현**,
잔차 부스팅 **하한 보장 성립**(Q4-K −0.0523 → **+0.0000**).

## ★★★ 이 런이 고치는 것 — **단변량 최고인데 증분이 0 인 역설**

```
prevTP_energy  단변량 0.2875 ← 지금까지 잰 것 중 **최고**   |ρ| vs base 0.4830 ← 표에서 **최고**
prevT_late     단변량 0.2664                              |ρ| vs base 0.3985
clus_d_own     단변량 0.2725                              |ρ| vs base 0.2562
(형태 최고 p_energy_ratio 0.2361)
```

### 원인 ① — **중복 감사가 `base` 만 봤다. 그런데 팔은 `morph` 위에 얹었다**

`rank_dup()` 은 `F_BASE`(RR 9열)에만 대고 잰다. 정작 대비는 `morph + 새열` vs
`morph + 셔플` 인데 **`MORPH` 와의 중복은 한 번도 안 쟀다.** 관문 통과 조건이
자기 팔과 무관한 것을 재고 있었다.

### 원인 ② — **`prevTP_energy` 의 창이 RR 에 따라 현재 박동 위로 미끄러진다** (산술)

박동 창은 R 이 index 100, 360Hz, 길이 300 → **−277.8 ~ +552.8ms**.
`WTP = (265, 300)` = **직전 R 기준 +458.3 ~ +555.6ms**. 그러면 **현재 R 기준** 위치는:

```
RR(ms)   현재 R 기준 창 위치       실제로 재는 것
 900     −441.7 ~ −344.4         T-P 기저선  ← 설계 의도
 800     −341.7 ~ −244.4         T-P 기저선
 700     −241.7 ~ −144.4         ★ 현재 박동의 **P 파**
 583     −124.7 ~  −27.4         ★ 현재 **P + QRS 시작**(q_on 중앙 −27.8ms)
 486      −27.7 ~  +69.6         ★ 현재 **QRS/ST**
```

⇒ **긴 RR 에선 기저선, 짧은 RR 에선 현재 QRS 를 읽는다.** 즉 이 열은
**「RR 로 게이팅된 형태 재독(再讀)」** 이다. RR 은 `base` 에, 형태는 `morph` 에
**이미 둘 다 있다**. 단변량이 최고인 것도(둘의 곱이니까), 증분이 0 인 것도
(둘 다 이미 있으니까) **같은 하나의 사실**이다. `prevT_late`(+333~+458ms)도 같은
기전이 약하게 걸린다 — |ρ| 순서가 정확히 그 순서다.

### 원인 ③ — **군집은 팔 안의 특징 공간에서 군집했다**

`clus_feats(np.c_[MORPH, INTV])` — 거리 좌표계가 **이미 팔에 들어 있는 열들**이다.
`clus_d_own`·`clus_d_major` 는 그 열들의 (비선형이지만) 결정론적 함수다.
단변량이 높은 것도(잘 되는 특징의 요약이니까), 증분이 0 인 것도 같은 이유다.

## ★★★ 딥러닝이 **아직도 못 올라오는** 구조적 이유 — α=0 은 결과가 아니라 **버그**다

Q4-M 은 5겹 전부에서 **α = +0.0000**, Δ = **+0.0000** 이었다. 「CNN 이 더할 게 없다」로
읽었는데, **학습이 시작조차 못 한 것**이다.

```python
self.alpha = nn.Parameter(torch.zeros(1))                    # α = 0
nn.init.zeros_(self.h.weight); nn.init.zeros_(self.h.bias)   # h = 0   ← 여기
```

`logit = off + α·h(z)` 에서

```
∂L/∂α   = (∂L/∂logit) · h(z)      h=0 이면 **0**
∂L/∂h_w = (∂L/∂logit) · α  · z    α=0 이면 **0**
```

**둘이 서로를 0 에 가둔다.** 국소 수치 확인(같은 구조·합성 데이터):

```
h=0     · α=0     ← Q4-M 설정   학습된 α +0.0000 · AUROC 0.7333 → 0.7333 (Δ +0.0000)
h=정상  · α=0     ← 고친 설정   학습된 α **+0.8349** · AUROC 0.7333 → **0.7920** (Δ +0.0587)
h=정상  · α=0.1                학습된 α +0.8968 · AUROC 0.7333 → 0.7920 (Δ +0.0587)
```

α=0 **출발(하한 보장)** 은 유지하되 `h` 만 정상 초기화하면 된다.

### 그리고 **손실 함수가 지표와 다르다**

우리 지표는 **레코드 안에서 상위 k 개**다. 그런데 BCE 는 **풀링 로그우도**를
최적화한다 — 레코드 간 절대 눈금을 맞추는 데 용량을 쓰고, 레코드 **안의 순서**에는
직접적인 압력이 없다. Q4-E/H2 에서 우리가 이미 증명한 것과 정확히 같은 어긋남이다
(레코드별 상수 시프트는 레코드 내 지표를 **정확히 0.0e+00** 만큼도 못 바꾼다).
⇒ **레코드 내 pairwise 순위 손실**을 붙여 목적함수를 지표에 맞춘다.

## ★★★ 문헌이 바꾼 것 — **범위(scope)와 운영점**

선생님이 주신 자료의 두 가지가 우리 설계를 직접 건드린다.

**① AAMI EC57 — AF 구간은 SVEB 평가에서 아예 제외한다.** MIT-BIH 는 AF 안에서
A/a 를 안 매기고(ESC 2020: AF = P 파의 완전 소실), 그래서 우리가 Q4-J 에서 「FN 의
90%가 AF 대리 상위 절반에 몰려 있다」고 읽은 것은 **모델 실패가 아니라 채점 범위
오류**일 수 있다. 이번에 **AF 구간을 분자·분모 양쪽에서 빼고** 다시 잰다.
⚠️ 리듬 라벨이 없으므로(Q7-D) **무라벨 검출기**를 만들고, **「검출 구간 안에서 S
밀도가 실제로 떨어지는가」** 를 타당성 관문으로 건다. 안 떨어지면 **제외하지 않는다**.

**② 운영점이 한 자릿수 틀렸다.** 문헌 판독 운영점은 **환자당 상위 30~50 박동**이고
(Top-50 에서 burden 추정 정확도 ≥95% 유지 · 판독시간 ≥80% 절감), 우리는 **k=300** 을
써 왔다. 그리고 형태의 이득은 **k 가 작을수록 크다**(k=50 +0.2230 → k=300 +0.0950).
⇒ `K_LIT = (10, 20, 30, 50)` 을 병기한다.

**③ 진짜 임상 표적은 환자 단위일 수 있다 — ESVEA.** ≥30 PAC/시간(또는 ≥20박 연속)
이면 신규 AF **HR 2.78** · 뇌졸중 **HR 2.40**. SVDB 는 레코드당 ~30분이므로
**S ≥ 15개 ⇔ ≥30 PAC/hour**. 박동 단위가 포화라도 **환자 단위 분류**는 아직 안 쟀다.

## 이번에 넣는 새 축 — **P 벡터축**(2리드를 처음으로 함께 쓴다)

지금까지 모든 P 특징은 **한 리드**(`lp`)만 봤다. 그런데 이소성 P′ 의 임상 기준은
**축 이동 ≥30°** 이고, 축은 정의상 **리드가 둘 이상** 있어야 잰다. SVDB 는 2리드다.

```
p_axis_dev     2리드 P 벡터각과 템플릿 벡터각의 차(도)      ← 임상 기준 그 자체
p_axis_f30     ≥30° 플래그
p_pol_ndiff    템플릿 대비 극성이 뒤집힌 리드 수 (0/1/2)   ← 리드별 극성 조합
p_vec_mag      |P 벡터| / 템플릿 |P 벡터|
p_corr_other   **버려 온 반대 리드**에서의 P |corr|
qrs_axis_dev   같은 벡터각을 **QRS** 에 — S 는 QRS 축이 정상, V 는 크게 튄다
```

## 그리고 **P-on-T 를 RR 정규화해서 다시 시험한다**

원인 ②가 진단이면 처방은 명확하다 — 창을 **RR 의 분수**로 잡아 **현재 R 을 절대
넘지 않게** 한다: 직전 R + `0.35·RR` ~ `min(0.95·RR, 창끝)`, **고정 32표본으로 리샘플**
(모양만 본다), 그 뒤 레코드 안에서 **RR 로 잔차화**한다.
사전등록 예측: `|ρ| vs base` 가 0.4830 → **0.30 미만**으로 떨어져야 한다.
안 떨어지면 진단이 틀린 것이고, 그때는 P-on-T 를 접는다.

## 관문 (사전등록)

| 관문 | 무엇 | 통과 기준 |
|---|---|---|
| **R0** | 자 검증(Q4-M 과 동일) | 깨지면 **중단** |
| **R1** | **중복 감사 v3** — `base` 가 아니라 **`base+morph` 전체**에 대고 잰다 + Q4-M 회고 | 입구 검사 |
| **R2 ★★★ 주 관문** | `vec` vs `vshuf` — **형태 위에** P/QRS 벡터축 6열 | 측정된 raw 영점 상단 초과 |
| **R3 ★★** | `pont2` vs `p2shuf` — RR 정규화 P-on-T. **먼저 `\|ρ\|`가 0.30 미만으로 떨어져야** 한다 | 진단 검증 + 증분 |
| **R4 ★★★** | **AAMI 범위** — AF 구간 무라벨 검출 → **S 밀도 하락**이 확인되면 분자·분모에서 제외 | 타당성 관문(성능 관문 아님) |
| **R5 ★★** | **운영점** `K_LIT=(10,20,30,50)` · **ESVEA 환자 단위 분류** | 측정 |
| **R6 ★★** | **딥러닝 3변형** — 데드락 재현 / 초기화 수정 / **레코드 내 순위손실** | 관문 아님 · 하한 보장 |

⚠️ **새 데이터 0** · **리듬 라벨은 없다**(Q7-D 확정 — AF 는 무라벨 검출기 + 타당성 검사).


In [ ]:
# CELL 0 — 공용 사전점검
import numpy as np

def decide(lo, hi, thr, direction):
    if direction not in (">", "<"):
        raise ValueError("direction 은 '>' 또는 '<'")
    if not (np.isfinite(lo) and np.isfinite(hi) and np.isfinite(thr)):
        return "⚠️ 미결"
    if direction == ">":
        if lo > thr: return "✅ 지지"
        if hi < thr: return "❌ 기각"
    else:
        if hi < thr: return "✅ 지지"
        if lo > thr: return "❌ 기각"
    return "⚠️ 미결"

def mde(lo, hi):
    return (hi - lo) / 2.0 if np.isfinite(lo) and np.isfinite(hi) else float("nan")

def boot_mean(v, seed, nb=3000, q=2.5):
    d = np.asarray(v, float); d = d[np.isfinite(d)]
    if len(d) < 3:
        return float("nan"), float("nan"), float("nan"), len(d)
    rng = np.random.RandomState(seed)
    b = [d[rng.randint(0, len(d), len(d))].mean() for _ in range(nb)]
    return (float(d.mean()), float(np.percentile(b, q)),
            float(np.percentile(b, 100 - q)), len(d))

def boot_pair(a, b, seed, nb=3000, q=2.5):
    a = np.asarray(a, float); b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b); a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan"), float("nan"), float("nan"), len(a)
    rng = np.random.RandomState(seed)
    d = [(b[j] - a[j]).mean() for j in (rng.randint(0, len(a), len(a)) for _ in range(nb))]
    return (float((b - a).mean()), float(np.percentile(d, q)),
            float(np.percentile(d, 100 - q)), len(a))

def need_super(n, half, eff, p80=False):
    if not np.isfinite(half) or not np.isfinite(eff) or abs(eff) < 1e-9 or n < 1:
        return float("nan")
    r = float(n) * (half / abs(eff)) ** 2
    return r * 2.04 if p80 else r

class AssetError(RuntimeError): pass
print("CELL 0 ✅")


In [ ]:
# CELL 1 — 설정 · 사전등록
import os, sys, json, importlib, time, warnings
importlib.invalidate_caches(); warnings.filterwarnings("ignore")

SMOKE = os.environ.get("MEDKOS_SMOKE") == "1"
_ENV_ROOT = os.environ.get("MEDKOS_DRIVE_ROOT")
if _ENV_ROOT:
    DRIVE_ROOT = _ENV_ROOT
else:
    try:
        from google.colab import drive; drive.mount("/content/drive", force_remount=False)
        DRIVE_ROOT = "/content/drive/MyDrive"
    except Exception as e:
        print("Colab 아님:", e); DRIVE_ROOT = "/content"
PROJECT = os.path.join(DRIVE_ROOT, "MedKOS", "ecg-model")
MITBIH  = os.path.join(DRIVE_ROOT, "mitbih")
sys.path.insert(0, os.path.join(PROJECT, "lib"))
from medkos_run import MedKOSRun

SEED0, IDX_S = 20260806, 1
RHY_K = (5, 10, 20, 32)
MIN_S, MIN_N = 25, 25
DEV_EVERY = 4
K_SWEEP = (50, 100, 200, 300)        # ★★★ 주 지표 = 이 네 점의 평균 달성률(Q4-M 과 동일)
K_LIT   = (10, 20, 30, 50)           # ★★★ 문헌 판독 운영점(환자당 상위 30~50 박동)
MAIN_K = 300
MAX_NEG_SLOPE, MIN_AUC_SLOPE = 0.10, 0.55
FS = 360.0
R_IDX = 100
W_P_S  = (20, 88)
W_T_S  = (135, 265)
W_Q_S  = (72, 148)
FRAC_QRS, FRAC_P, FRAC_T = 0.10, 0.25, 0.25
LAG = 30
P_CORR_MIN = 0.35
P_GAP = 4
P_NULL_Q = 0.95
TMPL_LO, TMPL_HI, TMPL_MIN = 0.92, 1.08, 30
DUP_RHO = 0.85                       # ★ 중복 문턱(사전 고정 · Q4-M 과 동일)
DUP_FULL_WARN = 0.60                 # ★★★ v3 — `base+morph` 전체 대비 경고선(사전 고정)
PONT2_RHO_MAX = 0.30                 # ★★★ R3 1단계 — RR 누출이 실제로 빠졌는가(사전 고정)
QRS_LO_MS, QRS_HI_MS = 70.0, 130.0
PR_LO_MS, PR_HI_MS = 110.0, 230.0
PW_LO_MS, PW_HI_MS = 60.0, 130.0
NB_BOOT = 400 if SMOKE else 2000
N_PERM  = 2   if SMOKE else 6

# ── ★★★ RR 정규화 P-on-T (원인 ② 처방)
PT_LO_F, PT_HI_F, PT_NRS = 0.35, 0.95, 32   # 직전 R + 0.35RR ~ min(0.95RR, 창끝) → 32표본
AXIS_DEG = 30.0                              # ★ 임상 기준 — P 축 이동 ≥30°

# ── ★★★ AAMI EC57 범위 — AF 구간 무라벨 검출(리듬 라벨이 없다 · Q7-D)
AF_WIN, AF_STEP = 32, 8      # 박동 단위 슬라이딩 창
AF_RMSSD, AF_PFOUND = 0.12, 0.60   # 불규칙성 상한 / P 검출률 하한(사전 고정 · 절대값)
AF_MIN_FRAC = 0.02           # 이만큼도 안 잡히면 「검출기가 안 돈다」로 본다
ESVEA_PER_HOUR = 30.0        # ★ ESVEA 기준 — ≥30 PAC/시간

ARMS = ("base", "morph", "vec", "vshuf", "pont2", "p2shuf", "comb")
MAIN_CT = "vec-vshuf"
CONTRASTS = (("morph-base",     "base",   "morph"),
             ("vec-vshuf",      "vshuf",  "vec"),
             ("vec-morph",      "morph",  "vec"),
             ("pont2-p2shuf",   "p2shuf", "pont2"),
             ("pont2-morph",    "morph",  "pont2"),
             ("comb-morph",     "morph",  "comb"))
# ── ★★ 딥러닝 3변형 — 데드락 재현 / 초기화 수정 / 레코드 내 순위손실
DL_FOLDS, DL_BATCH, DL_EMB, DL_WD = 5, 1024, 16, 1e-4
DL_EPOCH = 3 if SMOKE else 12   # ★ 스모크는 비용 손잡이만 축소
DL_VARIANTS = (("boost_dead", "zeros", "bce"),    # ★ Q4-M 재현 — α 가 0 에 갇혀야 한다
                ("boost_fix",  "normal", "bce"),   # ★★★ 초기화만 고친다
                ("boost_rank", "normal", "rank"))  # ★★★ 목적함수를 지표에 맞춘다
DL_PAIRS = 4096              # 순위손실 — 미니배치당 레코드 내 (양성, 음성) 쌍 수
PRIMARY, SECONDARY = "ksw", "auc"
READ_ORDER = ("R0", "R1", "R2", "R3", "R4", "R5", "R6")
SV5 = os.path.join(MITBIH, "svdb_data5.npz")

REF = dict(
    q4m=dict(ksw=dict(base=0.6791, morph=0.8361, pont=0.8293, pshuf=0.8360,
                      clus=0.8307, cshuf=0.8356, comb=0.8231),
             ach=dict(base=0.8074, morph=0.9018, comb=0.8860),
             auc=dict(base=0.9420, morph=0.9529, comb=0.9463),
             gates=dict(morph_base=0.1570, pont_pshuf=-0.0067, clus_cshuf=-0.0049,
                        comb_morph=-0.0130),
             uni=dict(prevTP_energy=0.2875, clus_d_own=0.2725, prevT_late=0.2664,
                      p_energy_ratio=0.2361),
             rho_base=dict(prevTP_energy=0.4830, prevT_late=0.3985, clus_d_own=0.2562),
             qrs_ms=77.8, pr_ms=131.9, pw_ms=75.0, inv=0.202, pf_n=0.954, pf_v=0.877,
             lead_split=0.641, p_thr=0.464, bite=0.218,
             fp_v=dict(base=4380, morph=1444, pont=1426, clus=1368, comb=1322),
             k_gain=dict(k50=0.2230, k100=0.1810, k200=0.1300, k300=0.0950),
             boost=dict(alpha=0.0, gain=0.0)),
    q4k=dict(qrs_ms=44.4, pr_ms=104.2, pw_ms=75.0, p_found=0.984,
             tp_r2=0.020, tp_uni=0.4269, base_uni=0.4337),
    # ★ 초기화 데드락 국소 수치 확인(같은 구조 · 합성)
    dead=dict(zeros=dict(alpha=0.0000, auc0=0.7333, auc1=0.7333),
              normal=dict(alpha=0.8349, auc0=0.7333, auc1=0.7920)),
    lit=[("de Chazal 2004", 0.759, 0.385), ("Llamedo 2011", 0.77, 0.39),
         ("1D CNN inter-patient", 0.7456, None),
         ("ESVEA Binici 2010", "신규 AF HR 2.78 · 뇌졸중 HR 2.40", ">=30 PAC/h")])

RULE_CHECK = {
    "R11 매크로":       "환자 단위 · 상한과 함께 읽는다",
    "R16 fallback 없음": "**자 검증(R0)이 깨지면 중단**한다",
    "R22 누출 없음":     "LORO · 델리네이션·템플릿·군집·AF 검출은 **라벨을 안 쓴다**. `sym` 은 "
                        "**자 검증과 사후 구성분석에만**",
    "R26 영점":         "영점은 raw(비교정) · rep 수준 산포 병기 · 흐리면 기각을 미결로 강등",
    "R29 ② 분기 금지":   "R0 이 깨지면 아래를 **안 읽는다**",
    "R33 ① MDE":        "관문마다 MDE. 미결 ≠ 등가",
    "R34 ② 문턱 금지":  "창·임계·k-스윕·`K_LIT`·중복문턱·AF 문턱을 **데이터 보기 전** 고정",
    "R35 ① 자 먼저":    "★★★ **중복 감사를 자기 팔에 맞춘다** — Q4-M 은 `base` 에만 대고 쟀는데 "
                        "팔은 `morph` 위에 얹었다. 관문이 자기와 무관한 것을 재고 있었다",
    "R36 ⑤ 자기검사":   "★★★ **데드락을 재현 팔로 증명**한다 — `boost_dead` 의 α 가 0 에 안 "
                        "갇히면 내 진단이 틀린 것이다",
    "R38 ⑦ 요약 정합":  "★★★ **AF 제외는 성능 관문이 아니라 타당성 관문**이다 — 분모가 줄면 수는 "
                        "그냥 오른다. 제외의 **정당성**만 판정하고 성능은 병기한다",
    "R39 ① 대안설명":   "RR 누출을 빼고도 P-on-T 가 안 얹히면 **그때가 진짜 기각**이다",
    "R40 ② 대조 설계":  "★★ `vshuf`·`p2shuf` 는 **차원 동일 · 레코드 내 행 치환**(내용만 제거)",
    "R41 ② 0 근처":     "효과가 0 근처면 필요표본은 해석 불가",
}

CONFIG = dict(
    exp="quest46_q4n_scope_rank_vector", quest="ailab-2026-0046", step="scope-rank-vector",
    parent_exp=["quest46_q4m_pont_cluster_boost"],
    purpose=("★★★ **Q4-M 의 역설을 푼다 — 단변량 최고인 열들의 증분이 0 이었다.** "
             "`prevTP_energy` 단변량 **0.2875**(측정 이래 최고 · 형태 최고 0.2361 보다 높다)인데 "
             "`pont − pshuf` 는 **−0.0067** 이다. 원인 셋을 특정했다. "
             "① **중복 감사가 `F_BASE`(RR 9열)에만 대고 쟀다** — 정작 팔은 `morph` 위에 얹었는데 "
             "**`MORPH` 와의 중복은 한 번도 안 쟀다**. 관문이 자기와 무관한 것을 재고 있었다. "
             "⇒ **v3: `base+morph` 전체에 대고 잰다**. "
             "② **`prevTP_energy` 의 창이 RR 에 따라 현재 박동 위로 미끄러진다**(산술). 창 "
             "`(265,300)` = 직전 R +458.3~+555.6ms 인데, 현재 R 기준으로는 RR 900ms 에서 "
             "−441.7~−344.4ms(=T-P 기저선 · 의도대로)지만 RR **700ms 에서 현재 P 파**, "
             "RR **583ms 에서 현재 P+QRS 시작**, RR **486ms 에서 현재 QRS/ST** 를 읽는다. "
             "즉 이 열은 **「RR 로 게이팅된 형태 재독」** 이고, RR 은 `base` 에 형태는 `morph` 에 "
             "**이미 둘 다 있다** — 단변량 최고인 것도 증분 0 인 것도 **같은 하나의 사실**이다. "
             "`|ρ| vs base` 가 표에서 최고(0.4830)이고 `prevT_late`(0.3985)가 둘째인 것이 정확히 "
             "그 순서다. ⇒ **RR 정규화**(직전 R + 0.35RR ~ min(0.95RR, 창끝) · 32표본 리샘플 · "
             "RR 잔차화)로 다시 시험하되, **`|ρ|` 가 0.30 미만으로 떨어지는지를 1단계 관문**으로 "
             "건다(안 떨어지면 진단이 틀린 것이고 P-on-T 를 접는다). "
             "③ **군집이 팔 안의 특징 공간에서 군집했다** — `clus_feats(np.c_[MORPH, INTV])` 라 "
             "거리 좌표계가 이미 팔에 있는 열들이다. ⇒ 이번엔 **뺀다**. "
             "★★★ **딥러닝의 α=0 은 결과가 아니라 초기화 데드락이다.** "
             "`logit = off + α·h(z)` 에서 `∂L/∂α = (∂L/∂logit)·h(z)` 이고 "
             "`∂L/∂h_w = (∂L/∂logit)·α·z` 인데 **α 와 h 를 둘 다 0 으로 초기화**했다 — "
             "서로를 0 에 가둔다. 국소 확인: h=0·α=0 → α +0.0000 · Δ +0.0000, "
             "h=정상·α=0 → α **+0.8349** · AUROC 0.7333 → **0.7920**. ⇒ α=0 출발(하한 보장)은 "
             "유지하고 **h 만 정상 초기화**한다. 그리고 **손실이 지표와 다르다** — 지표는 "
             "레코드 내 상위 k 인데 BCE 는 풀링 로그우도다(Q4-E H2: 레코드별 상수 시프트는 "
             "레코드 내 지표를 0.0e+00 도 못 바꾼다). ⇒ **레코드 내 pairwise 순위손실**을 붙인다. "
             "**3변형으로 판정**한다 — 데드락 재현 / 초기화 수정 / 순위손실. "
             "★★★ **문헌이 범위와 운영점을 바꿨다.** ① **AAMI EC57 — AF 구간은 SVEB 평가에서 "
             "제외**한다(MIT-BIH 는 AF 안에서 A/a 를 안 매긴다 · ESC 2020: AF = P 파 완전 소실). "
             "리듬 라벨이 없으므로 **무라벨 검출기**를 만들고 **「검출 구간 안에서 S 밀도가 실제로 "
             "떨어지는가」** 를 타당성 관문으로 건다 — 안 떨어지면 **제외하지 않는다**. "
             "② **운영점이 한 자릿수 틀렸다** — 문헌은 **환자당 상위 30~50 박동**(Top-50 에서 "
             "burden 정확도 ≥95% 유지 · 판독시간 ≥80% 절감)인데 우리는 k=300 을 썼고, 형태의 "
             "이득은 k 가 작을수록 크다(k=50 +0.2230 → k=300 +0.0950). ⇒ `K_LIT=(10,20,30,50)` 병기. "
             "③ **ESVEA**(≥30 PAC/시간 → 신규 AF HR 2.78 · 뇌졸중 HR 2.40)를 **환자 단위 표적**으로 "
             "처음 잰다. "
             "★★ **새 축은 P 벡터축** — 지금까지 모든 P 특징이 **한 리드**(`lp`)만 봤는데, "
             "이소성 P′ 의 임상 기준은 **축 이동 ≥30°** 이고 축은 리드가 둘 이상 있어야 잰다. "
             "SVDB 는 2리드다 — **한 번도 함께 쓴 적이 없다**."),
    dataset="SVDB — svdb_data5.npz (새 데이터 0)",
    arms=list(ARMS), main_contrast=MAIN_CT, primary=PRIMARY, secondary=SECONDARY,
    k_sweep=list(K_SWEEP), k_lit=list(K_LIT), fs=FS, r_idx=R_IDX,
    windows=dict(p=W_P_S, t=W_T_S, q=W_Q_S, frac=(FRAC_QRS, FRAC_P, FRAC_T), lag=LAG,
                 p_corr_min=P_CORR_MIN, pont2=(PT_LO_F, PT_HI_F, PT_NRS), axis_deg=AXIS_DEG),
    guards=dict(dup_rho=DUP_RHO, dup_full_warn=DUP_FULL_WARN, pont2_rho_max=PONT2_RHO_MAX,
                qrs=(QRS_LO_MS, QRS_HI_MS), pr=(PR_LO_MS, PR_HI_MS),
                af=(AF_WIN, AF_STEP, AF_RMSSD, AF_PFOUND, AF_MIN_FRAC),
                esvea_per_hour=ESVEA_PER_HOUR),
    read_order=READ_ORDER, dev_every=DEV_EVERY, n_boot=NB_BOOT, n_perm=N_PERM,
    smoke=SMOKE, ref=REF, rule_check=RULE_CHECK,
    predictions={
        "R0": "★★★ **자 검증(중단 관문)** — Q4-M 과 동일 기준. QRS 70~130ms · PR 110~230ms · "
              "P 폭 60~130ms · 역위 P > 0 · **V 에서 |corr|·검출률이 N 보다 낮아야 한다**",
        "R1": "★★★ **중복 감사 v3** — `base` 가 아니라 **`base+morph` 전체**에 대고 잰다. "
              "Q4-M 의 `prevTP_energy` 를 **회고 재판정**하고, **창 미끄러짐**을 `pre_rr` 로 "
              "직접 계량한다(현재 P 시작·현재 R 을 넘는 박동 비율)",
        "R2": "★★★ **주 관문** — `vec` vs `vshuf`(차원 동일) · **형태 위에** P/QRS 벡터축 6열. "
              "2리드를 함께 쓰는 첫 특징이다",
        "R3": "★★ **2단 관문** — ① RR 정규화로 `|ρ| vs base` 가 **0.30 미만**으로 떨어지는가"
              "(진단 검증) ② 그러고도 형태 위에 얹히는가",
        "R4": "★★★ **AAMI 범위** — AF 구간 무라벨 검출. **타당성 관문**: 검출 구간 안 S 밀도가 "
              "바깥보다 **낮아야** 제외한다(MIT-BIH 는 AF 안에서 A/a 를 안 매긴다). "
              "성능은 병기만 한다 — 분모가 줄면 수는 그냥 오른다",
        "R5": "★★ **운영점** `K_LIT=(10,20,30,50)` · **ESVEA**(≥30 PAC/h ⇔ 30분 레코드에서 "
              "S ≥ 15) 환자 단위 분류 AUROC",
        "R6": "★★ **딥러닝 3변형** — `boost_dead`(재현 · α 가 0 에 갇혀야 한다) / "
              "`boost_fix`(h 정상 초기화) / `boost_rank`(레코드 내 순위손실). "
              "셋 다 **α=0 출발**이라 하한이 보장된다"},
    caveat=("★★★ **이 런의 절반은 자를 고치는 런이다**(R35 ①) — Q4-M 의 중복 감사는 "
            "`F_BASE` 에만 대고 쟀는데 팔은 `morph` 위에 얹었다. 그리고 `prevTP_energy` 의 "
            "창 미끄러짐은 **가설이 아니라 산술**이다(창 정의 + `pre_rr` 로 이 런이 직접 계량한다). "
            "★★ **AF 제외는 성능 주장이 아니다** — 분모에서 빼면 지표는 기계적으로 오른다. "
            "이 런이 판정하는 것은 **제외의 정당성**(검출 구간 안 S 밀도 하락)뿐이고, "
            "성능은 범위를 밝혀 병기한다(R38 ⑦). 검출기가 거의 아무것도 안 잡거나 S 밀도가 "
            "안 떨어지면 **제외하지 않는다**. "
            "★★ **딥러닝 결론을 뒤집을 수 있다** — Q4-M 의 「α=0 · CNN 이 더할 게 없다」는 "
            "**초기화 데드락**이었다. `boost_dead` 가 α=0 을 재현하고 `boost_fix` 가 안 그러면 "
            "진단이 확정되고, 그때 Q4-M 의 P5 해석은 **철회**된다. "
            "★ **군집 축은 이번에 뺀다** — 팔 안의 특징 공간에서 군집했으므로 다시 얹어도 "
            "같은 결과가 나온다. 되살리려면 **팔 밖의 좌표계**(원파형 잔차)가 필요하다."))
np.random.seed(SEED0)
run = MedKOSRun("quest46_q4n_scope_rank_vector", CONFIG, project=PROJECT)
run.log("설정 ✅ **Q4-N — 범위·운영점·P 벡터축·순위손실**")
run.log("  ★★★ Q4-M 의 역설 — **단변량 최고인 열의 증분이 0** 이었다")
run.log(f"     `prevTP_energy` 단변량 {REF['q4m']['uni']['prevTP_energy']} "
        f"(형태 최고 {REF['q4m']['uni']['p_energy_ratio']} 보다 높다) → "
        f"`pont−pshuf` {REF['q4m']['gates']['pont_pshuf']:+.4f}")
run.log("     ① 중복 감사가 **`F_BASE` 에만** 대고 쟀다 — 팔은 `morph` 위에 얹었는데")
run.log("     ② `prevTP_energy` 창이 **RR 따라 현재 박동 위로 미끄러진다**(산술 · 이 런이 계량)")
run.log("     ③ 군집이 **팔 안의 특징 공간**에서 군집했다 → 이번엔 뺀다")
run.log("  ★★★ 딥러닝 α=0 은 결과가 아니라 **초기화 데드락**이다")
run.log(f"     h=0·α=0 → α {REF['dead']['zeros']['alpha']:+.4f} · AUROC "
        f"{REF['dead']['zeros']['auc0']} → {REF['dead']['zeros']['auc1']}")
run.log(f"     h=정상·α=0 → α **{REF['dead']['normal']['alpha']:+.4f}** · AUROC "
        f"{REF['dead']['normal']['auc0']} → **{REF['dead']['normal']['auc1']}**")
run.log(f"  ★★★ 문헌 — **AAMI EC57: AF 구간 제외** · 운영점 **환자당 상위 30~50**"
        f"(우리는 k=300 이었다) · **ESVEA** ≥{ESVEA_PER_HOUR:.0f} PAC/h")
run.log(f"  ⇒ 주 지표는 k-스윕 {list(K_SWEEP)} 유지 · 문헌 운영점 {list(K_LIT)} **병기**")
if SMOKE:
    run.log(f"  ⚠️ **스모크런** — 비용 손잡이만 축소(NB_BOOT={NB_BOOT} · N_PERM={N_PERM})")
run.log("\n  사전등록 규칙 체크리스트 (R29 ③)")
for k_, v_ in RULE_CHECK.items():
    run.log(f"    [x] {k_:<18} {v_}")


In [ ]:
# CELL 2 — 【P-0】 코호트 · ★★★ 고친 델리네이션 · 특징
import pandas as pd
from collections import Counter
from scipy.stats import spearmanr, rankdata
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import average_precision_score, roc_auc_score

run.log("\n" + "=" * 100)
run.log("【P-0】 ★★★ 고친 델리네이션 — 역위 P · QRS 3상 · 리드 분리")
run.log("=" * 100)
VERD, NOTE = {}, {}
def g_(k, v, d):
    VERD[k] = v; NOTE[k] = d; run.log(f"  {k:<5}{v}  {d}")

if not os.path.exists(SV5):
    raise AssetError(f"{SV5} 없음(R16)")
D5 = np.load(SV5, allow_pickle=True)
for need in ("pid", "y3", "pre_rr", "post_rr", "beat", "sym"):
    if need not in D5.files:
        raise AssetError(f"`{need}` 가 자산에 없다(R16)")
PID = np.asarray(D5["pid"]).astype(int); Y3 = np.asarray(D5["y3"]).astype(int)
PRE = np.asarray(D5["pre_rr"], float); POST = np.asarray(D5["post_rr"], float)
K = np.where(Y3 >= 0)[0]
RID = PID[K]; TT_ = (Y3[K] == IDX_S)
pre = PRE[K].astype(float); post = POST[K].astype(float)
BEAT = np.asarray(np.asarray(D5["beat"])[K], dtype="float32")
SYM = np.asarray(D5["sym"]).astype("<U2")[K]
NL, LW = BEAT.shape[1], BEAT.shape[2]
if LW <= W_T_S[1]:
    raise AssetError(f"파형 길이 {LW} 가 T 창 {W_T_S} 보다 짧다(R34 ②)")
RS = np.array(sorted(set(RID.tolist())))
IDX_ALL = {int(r): np.where(RID == r)[0] for r in RS}
run.log(f"  파형 {BEAT.shape} · 리드 {NL}")

_S = pd.Series(pre); _G = _S.groupby(pd.Series(RID))
def local_base(k):
    r = np.asarray(_G.apply(lambda x: x.shift(1).rolling(k, min_periods=1).median())).astype(float)
    return np.where(np.isfinite(r), r, pre)
_med = _G.transform("median").to_numpy()
_std = _G.transform("std").to_numpy(); _mean = _G.transform("mean").to_numpy()
BASE12 = local_base(12); REL = pre / (BASE12 + 1e-9)
F_BASE = np.nan_to_num(np.c_[_med - pre,
                             np.column_stack([1.0 - pre / (local_base(k) + 1e-9) for k in RHY_K]),
                             post - pre, np.nan_to_num(_std / (_mean + 1e-9)),
                             np.log1p(np.clip(pre, 0, None)), np.log1p(np.clip(post, 0, None))],
                       nan=0.0, posinf=0.0, neginf=0.0)

def corr_to(x, t):
    xc = x - x.mean(-1, keepdims=True); tc = t - t.mean(-1, keepdims=True)
    return (xc * tc).sum(-1) / (np.sqrt((xc ** 2).sum(-1) * (tc ** 2).sum(-1)) + 1e-9)

def peak_of(x, lo, hi, base):
    seg = np.abs(x[lo:hi] - base); k = int(np.argmax(seg))
    return lo + k, float(seg[k])

# ★ 고친 폭 — 봉우리 연속 확장이 아니라 **창 안 첫~마지막 임계 교차**.
#   Q–R–S 3상의 0 교차에서 끊기지 않는다(Q4-K 버그 ②).
def cross_span(x, lo, hi, frac, base):
    seg = np.abs(x[lo:hi] - base); amp = float(seg.max())
    if amp <= 1e-9: return lo, hi - 1, amp
    w = np.where(seg > frac * amp)[0]
    return lo + int(w[0]), lo + int(w[-1]), amp

# ★★★ Q4-L 의 **과잉 수정**을 되돌린다 — 첫~마지막 교차는 **다상(QRS)** 에만 옳다.
#   P·T 는 **단상**이라 봉우리에서 바깥으로 **연속 확장**하는 게 맞다.
#   Q4-L 의 P 폭 138.9ms 는 창(189ms) 전체를 P 로 잡은 결과였다.
def peak_span(x, lo, hi, frac, base):
    seg = np.abs(x[lo:hi] - base); k = int(np.argmax(seg)); amp = float(seg[k])
    if amp <= 1e-9: return lo, hi - 1, amp
    m = seg > frac * amp
    i = k
    while i > 0 and m[i - 1]: i -= 1
    j = k
    while j < len(m) - 1 and m[j + 1]: j += 1
    return lo + i, lo + j, amp

def cross_span_batch(B, lo, hi, frac, base):
    seg = np.abs(B[:, :, lo:hi] - base[:, :, None])
    amp = seg.max(-1)
    m = seg > frac * amp[..., None]
    first = m.argmax(-1)
    last = m.shape[-1] - 1 - m[:, :, ::-1].argmax(-1)
    w = np.where(m.any(-1), last - first + 1, 0)
    return w.astype(float), amp, first + lo, last + lo

def peak_span_batch(B, lo, hi, frac, base):
    """단상(P·T)용 — 봉우리에서 바깥으로 **연속** 확장. 창 전체를 삼키지 않는다."""
    seg = np.abs(B[:, :, lo:hi] - base[:, :, None])
    amp = seg.max(-1); pk = seg.argmax(-1)
    m = seg > frac * amp[..., None]
    n, nl, W = m.shape
    ar = np.arange(W)[None, None, :]
    below = ~m
    left = np.where(below & (ar < pk[..., None]), ar, -1).max(-1) + 1
    right = np.where(below & (ar > pk[..., None]), ar, W).min(-1) - 1
    w = np.clip(right - left + 1, 0, W)
    return w.astype(float), amp, left + lo, right + lo

INAMES = ["pr_rel", "dpr_ms", "pr_dev_ms", "pp_over_rr", "pw_over_qrsw", "pw_over_tw",
          "tp_resid_z", "pw_rel", "p_corr_abs", "p_polarity", "p_amp_rel", "p_found"]
MNAMES = ["corr_qrs_min", "corr_qrs_mean", "corr_full_min", "corr_st_min",
          "qrs_width", "amp_ratio", "area_ratio", "p_energy_ratio"]
PNAMES = ["corr_qrs_l0", "corr_qrs_l1", "p_e_early", "p_e_late", "corr_st_early", "corr_t_peak"]
DELIN, TMPL, P_THR = {}, {}, {}

def build_templates():
    for r in RS:
        ii = IDX_ALL[int(r)]
        okm = (REL[ii] >= TMPL_LO) & (REL[ii] <= TMPL_HI)
        if int(okm.sum()) < TMPL_MIN: okm = np.ones(len(ii), bool)
        T = np.median(BEAT[ii][okm], axis=0).astype(float)
        iso = np.median(T[:, W_P_S[0]:W_P_S[0] + 8], axis=-1)
        # ★ 리드 분리 — QRS 는 진폭 최대, **P 는 P 창 진폭 최대**(Q4-K 버그 ④)
        # ★★★ **QRS 를 먼저** 잡고, P 창을 **q_on 앞으로 자동 제한**한다.
        #   Q4-M 1차 실행이 P 폭 33.3ms · PR 40.3ms 로 죽은 원인이 이것이다 —
        #   고정 창 (20, 88) 이 q_on(≈86) 을 물어 **QRS 상승부를 P 봉우리로 잡았다**
        #   (QRS 진폭은 P 의 ~10배다). 합성 확인: 봉우리 idx 87 → 45, PR 8.3 → 152.8ms.
        lq = int(np.argmax([peak_of(T[l], W_Q_S[0], W_Q_S[1], iso[l])[1] for l in range(NL)]))
        q_on, q_off, q_amp = cross_span(T[lq], W_Q_S[0], W_Q_S[1], FRAC_QRS, iso[lq])
        p_hi = int(max(W_P_S[0] + 14, min(W_P_S[1], q_on - P_GAP)))
        lp = int(np.argmax([peak_of(T[l], W_P_S[0], p_hi, iso[l])[1] for l in range(NL)]))
        p_on, p_off, p_amp = peak_span(T[lp], W_P_S[0], p_hi, FRAC_P, iso[lp])       # 단상
        t_on, t_off, t_amp = peak_span(T[lq], W_T_S[0], W_T_S[1], FRAC_T, iso[lq])   # 단상
        p_pk = peak_of(T[lp], W_P_S[0], p_hi, iso[lp])[0]
        TMPL[int(r)] = T
        DELIN[int(r)] = dict(lq=lq, lp=lp, iso=iso, p_hi=p_hi, q=(q_on, q_off),
                             p=(p_on, p_off, p_pk),
                             t=(t_on, t_off), p_amp=p_amp, q_amp=q_amp,
                             qrs_ms=(q_off - q_on + 1) / FS * 1000.0,
                             pw_ms=(p_off - p_on + 1) / FS * 1000.0,
                             pr_ms=(q_on - p_on) / FS * 1000.0)
build_templates()
QRS_MS = float(np.median([DELIN[int(r)]["qrs_ms"] for r in RS]))
PW_MS  = float(np.median([DELIN[int(r)]["pw_ms"] for r in RS]))
PR_MS  = float(np.median([DELIN[int(r)]["pr_ms"] for r in RS]))
LP_NE_LQ = float(np.mean([DELIN[int(r)]["lp"] != DELIN[int(r)]["lq"] for r in RS]))
run.log(f"  ★ 고친 자 — QRS 폭 중앙 **{QRS_MS:.1f}ms**(Q4-K {REF['q4k']['qrs_ms']}) · "
        f"P 폭 **{PW_MS:.1f}ms**(Q4-K {REF['q4k']['pw_ms']}) · "
        f"PR(P시작→QRS시작) **{PR_MS:.1f}ms**(Q4-K {REF['q4k']['pr_ms']})")
run.log(f"  ★ 리드 분리 — P 리드 ≠ QRS 리드인 레코드 **{LP_NE_LQ:.1%}**")
Q_ON = float(np.median([DELIN[int(r)]["q"][0] for r in RS]))
P_PK = float(np.median([DELIN[int(r)]["p"][2] for r in RS]))
P_ON = float(np.median([DELIN[int(r)]["p"][0] for r in RS]))
P_HI = float(np.median([DELIN[int(r)]["p_hi"] for r in RS]))
BITE = float(np.mean([DELIN[int(r)]["p"][2] > DELIN[int(r)]["q"][0] - 8 for r in RS]))
run.log(f"  ★ **기하 진단**(표본 index) — QRS 시작 q_on 중앙 **{Q_ON:.0f}** · P 창 상단 "
        f"{P_HI:.0f}(=q_on−{P_GAP}) · P 봉우리 **{P_PK:.0f}** · P 시작 {P_ON:.0f}")
run.log(f"    **P 봉우리가 QRS 시작 8표본 안에 든 레코드 {BITE:.1%}** — 높으면 창이 QRS 를 "
        f"물은 것이다(Q4-M 1차 실행이 그래서 죽었다: P 폭 33.3ms · PR 40.3ms)")
run.log(f"    ▸ 참고 — `morph` 의 `p_energy_ratio` 창은 (25, 75) 고정이다. q_on {Q_ON:.0f} "
        f"보다 {'앞이라 안전' if Q_ON >= 76 else '**뒤라 QRS 를 물 수 있다**'}")

def interval_feats():
    out = np.zeros((len(K), 12), float)
    for r in RS:
        ii = IDX_ALL[int(r)]; d = DELIN[int(r)]; T = TMPL[int(r)]
        lp, lq = d["lp"], d["lq"]
        p_on, p_off, p_pk = d["p"]; q_on, q_off = d["q"]; t_on, t_off = d["t"]
        p_hi = d["p_hi"]
        B = BEAT[ii]
        pw0 = max(3, p_off - p_on + 1)
        seg_t = T[lp, p_on:p_on + pw0][None, :]
        best_a = np.full(len(ii), -1.0); best_s = np.ones(len(ii)); best_l = np.zeros(len(ii), int)
        for lag in range(-LAG, LAG + 1):
            a, b = p_on + lag, p_on + lag + pw0
            if a < 0 or b > p_hi: continue          # ★ QRS 를 물지 않게 클램프
            c = corr_to(B[:, lp, a:b], seg_t)
            ac = np.abs(c)                        # ★★★ 역위 P 를 잡는다(Q4-K 버그 ①)
            upd = ac > best_a
            best_a[upd] = ac[upd]; best_s[upd] = np.sign(c[upd]); best_l[upd] = lag
        # ★★★ 검출 문턱을 **레코드별 영점으로 교정**한다.
        #   61개 지연 중 최대 |corr| 는 **선택 효과**로 잡음에서도 0.5 를 쉽게 넘는다.
        #   그래서 **표본 치환 템플릿**(모양은 완전히 파괴, 진폭분포는 보존)으로 같은 절차를
        #   돌려 그 95 백분위를 문턱으로 쓴다. `p_found` 가 그제서야 의미를 갖는다.
        #   ⚠️ 시간 역전은 안 쓴다 — **대칭 파형에서는 항등**이라 영점이 신호와 같아진다.
        seg_r = seg_t[:, np.random.RandomState(SEED0 + int(r)).permutation(pw0)]
        nul_a = np.full(len(ii), -1.0)
        for lag in range(-LAG, LAG + 1):
            a, b = p_on + lag, p_on + lag + pw0
            if a < 0 or b > p_hi: continue          # ★ 영점도 같은 창에서
            nul_a = np.maximum(nul_a, np.abs(corr_to(B[:, lp, a:b], seg_r)))
        thr_r = max(P_CORR_MIN, float(np.quantile(nul_a, P_NULL_Q)))
        P_THR[int(r)] = thr_r
        p_found = (best_a >= thr_r).astype(float)
        base_b = np.median(B[:, :, W_P_S[0]:W_P_S[0] + 8], axis=-1)
        pw, pa, p_first, _ = peak_span_batch(B, max(0, p_on - LAG),
                                             min(p_hi, p_off + LAG + 1), FRAC_P, base_b)
        qw, qa, q_first, _ = cross_span_batch(B, W_Q_S[0], W_Q_S[1], FRAC_QRS, base_b)
        tw, ta, _, t_last = peak_span_batch(B, W_T_S[0], W_T_S[1], FRAC_T, base_b)
        pw = pw[:, lp]; pa = pa[:, lp]
        qw = qw[:, lq]; tw = tw[:, lq]
        p_on_b = (p_on + best_l).astype(float)
        q_on_b = q_first[:, lq].astype(float)
        pr_ms = (q_on_b - p_on_b) / FS * 1000.0      # ★ P 시작 → QRS 시작
        ok_ = p_found > 0
        med_pr = float(np.median(pr_ms[ok_])) if ok_.any() else float(np.median(pr_ms))
        dpr = np.r_[0.0, np.diff(pr_ms)]
        rr_s = pre[ii] * FS
        t_last_prev = np.r_[float(t_off), t_last[:-1, lq].astype(float)]
        tp = rr_s + p_on_b - t_last_prev
        A = np.c_[np.ones(len(ii)), rr_s]
        coef, *_ = np.linalg.lstsq(A, tp, rcond=None)
        tp_res = tp - A @ coef
        med_pw = float(np.median(pw)) + 1e-9; med_pa = float(np.median(pa)) + 1e-9
        out[ii, 0] = pr_ms / (med_pr + 1e-9)
        out[ii, 1] = dpr
        out[ii, 2] = pr_ms - med_pr
        out[ii, 3] = 1.0 - (dpr / 1000.0 * FS) / (rr_s + 1e-9)
        out[ii, 4] = pw / (qw + 1e-9)
        out[ii, 5] = pw / (tw + 1e-9)
        out[ii, 6] = tp_res / (float(np.std(tp_res)) + 1e-9)
        out[ii, 7] = pw / med_pw
        out[ii, 8] = best_a
        out[ii, 9] = best_s                          # ★★★ 역위 P = 소견
        out[ii, 10] = pa / med_pa
        out[ii, 11] = p_found
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

def morph_feats(extended=False):
    ncol = 6 if extended else 8
    out = np.zeros((len(K), ncol), float)
    WQ, WF, WS, WP, WW = (85, 125), (60, 220), (130, 260), (25, 75), (80, 130)
    WSE, WTP, WPE, WPL = (128, 175), (175, 225), (25, 50), (50, 75)
    for r in RS:
        ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]
        if not extended:
            cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
            cf = corr_to(B[:, :, WF[0]:WF[1]], T[:, WF[0]:WF[1]])
            cs = corr_to(B[:, :, WS[0]:WS[1]], T[:, WS[0]:WS[1]])
            seg = B[:, :, WW[0]:WW[1]]; med = np.median(seg, axis=-1, keepdims=True)
            amp = np.abs(seg - med).max(-1, keepdims=True) + 1e-9
            wid = (np.abs(seg - med) > 0.5 * amp).mean(-1)
            q = B[:, :, WQ[0]:WQ[1]]; ptp = q.max(-1) - q.min(-1)
            tq = T[:, WQ[0]:WQ[1]]; tptp = float(np.mean(tq.max(-1) - tq.min(-1))) + 1e-9
            area = np.abs(q - np.median(q, axis=-1, keepdims=True)).sum(-1)
            tarea = float(np.mean(np.abs(tq - np.median(tq, axis=-1, keepdims=True)).sum(-1))) + 1e-9
            p = B[:, :, WP[0]:WP[1]]
            pe = np.sqrt(((p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
            tp_ = T[:, WP[0]:WP[1]]
            tpe = float(np.mean(np.sqrt(((tp_ - tp_.mean(-1, keepdims=True)) ** 2).mean(-1)))) + 1e-9
            out[ii, 0] = cq.min(1); out[ii, 1] = cq.mean(1); out[ii, 2] = cf.min(1)
            out[ii, 3] = cs.min(1); out[ii, 4] = wid.mean(1)
            out[ii, 5] = ptp.mean(1) / tptp; out[ii, 6] = area.mean(1) / tarea
            out[ii, 7] = pe.mean(1) / tpe
        else:
            cq = corr_to(B[:, :, WQ[0]:WQ[1]], T[:, WQ[0]:WQ[1]])
            out[ii, 0] = cq[:, 0]; out[ii, 1] = cq[:, min(1, NL - 1)]
            for c_, W_ in ((2, WPE), (3, WPL)):
                p = B[:, :, W_[0]:W_[1]]
                pe = np.sqrt(((p - p.mean(-1, keepdims=True)) ** 2).mean(-1))
                tp_ = T[:, W_[0]:W_[1]]
                tpe = float(np.mean(np.sqrt(((tp_ - tp_.mean(-1, keepdims=True)) ** 2)
                                            .mean(-1)))) + 1e-9
                out[ii, c_] = pe.mean(1) / tpe
            out[ii, 4] = corr_to(B[:, :, WSE[0]:WSE[1]], T[:, WSE[0]:WSE[1]]).min(1)
            out[ii, 5] = corr_to(B[:, :, WTP[0]:WTP[1]], T[:, WTP[0]:WTP[1]]).min(1)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

# ── ★★★ 축 A — **P 벡터축**: 2리드를 **처음으로 함께** 쓴다
#   지금까지 모든 P 특징은 한 리드(`lp`)만 봤다. 그런데 이소성 P′ 의 임상 기준은
#   **축 이동 ≥30°** 이고, 축은 정의상 리드가 둘 이상 있어야 잰다. SVDB 는 2리드다.
VECNAMES = ["p_axis_dev", "p_axis_f30", "p_pol_ndiff", "p_vec_mag",
            "p_corr_other", "qrs_axis_dev"]
def _ang_dev(vx, vy, tx, ty):
    """두 2D 벡터의 사잇각(도) — 부호 없이 0~180. 진폭에 불변."""
    n1 = np.sqrt(vx ** 2 + vy ** 2) + 1e-12
    n2 = float(np.sqrt(tx ** 2 + ty ** 2)) + 1e-12
    c = np.clip((vx * tx + vy * ty) / (n1 * n2), -1.0, 1.0)
    return np.degrees(np.arccos(c))

def vec_feats():
    out = np.zeros((len(K), 6), float)
    if NL < 2:
        return out                       # 리드가 하나면 축은 정의되지 않는다(R16)
    for r in RS:
        ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]; d = DELIN[int(r)]
        lp, lq = d["lp"], d["lq"]; iso = d["iso"]
        p_on, p_off, p_pk = d["p"]; q_on, q_off = d["q"]
        # ★ 벡터 성분 = **각 리드에서 P 봉우리 시점의 등전위 대비 편차**(부호 유지)
        pv = B[:, :, p_pk] - iso[None, :]
        tv = T[:, p_pk] - iso
        out[ii, 0] = _ang_dev(pv[:, 0], pv[:, 1], float(tv[0]), float(tv[1]))
        out[ii, 1] = (out[ii, 0] >= AXIS_DEG).astype(float)
        # ★ 리드별 극성 조합 — 템플릿 대비 부호가 뒤집힌 리드 수(0/1/2)
        out[ii, 2] = (np.sign(pv[:, :2]) != np.sign(tv[:2])[None, :]).sum(1).astype(float)
        tmag = float(np.sqrt(float(tv[0]) ** 2 + float(tv[1]) ** 2)) + 1e-9
        out[ii, 3] = np.sqrt(pv[:, 0] ** 2 + pv[:, 1] ** 2) / tmag
        # ★ **버려 온 반대 리드**에서의 P 상관(역위 포함 → |corr|)
        lo_ = 1 - lp if NL >= 2 else lp
        pw0 = max(3, p_off - p_on + 1)
        out[ii, 4] = np.abs(corr_to(B[:, lo_, p_on:p_on + pw0],
                                    T[lo_, p_on:p_on + pw0][None, :]))
        # ★ 같은 벡터각을 **QRS** 에 — S 는 QRS 축이 정상, V 는 크게 튄다
        q_pk = peak_of(T[lq], W_Q_S[0], W_Q_S[1], iso[lq])[0]
        qv = B[:, :, q_pk] - iso[None, :]
        tq = T[:, q_pk] - iso
        out[ii, 5] = _ang_dev(qv[:, 0], qv[:, 1], float(tq[0]), float(tq[1]))
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)

# ── ★★★ 축 B — **RR 정규화 P-on-T**(Q4-M 원인 ② 의 처방)
#   Q4-M 의 `WTP=(265,300)` 은 **직전 R 기준 고정 표본**이라 RR 이 짧아지면 창이
#   현재 박동의 P → QRS 위로 미끄러진다(RR 700ms 에서 현재 P · 486ms 에서 현재 QRS).
#   ⇒ 창을 **RR 의 분수**로 잡아 **현재 R 을 절대 넘지 않게** 하고(상한 0.95RR),
#     32표본으로 리샘플해 **모양만** 보고, 마지막에 레코드 안에서 **RR 로 잔차화**한다.
PT2NAMES = ["pt2_corr", "pt2_energy", "pt2_resid_max", "pt2_resid_pos", "pt2_late"]
def pont2_feats():
    out = np.zeros((len(K), 5), float)
    grid = np.linspace(0.0, 1.0, PT_NRS)
    for r in RS:
        ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]; lq = DELIN[int(r)]["lq"]
        rr_s = np.clip(pre[ii] * FS, 60.0, 3.0 * FS)          # 표본 단위 RR
        # ★ 창 끝은 **현재 R 을 절대 안 넘고**(0.95RR) 파형 끝도 안 넘는다.
        #   긴 RR(>1.4초)에서는 시작점도 창 안으로 당겨야 창이 무너지지 않는다.
        a_f = np.minimum(R_IDX + PT_LO_F * rr_s, float(LW - 24))
        b_f = np.clip(R_IDX + PT_HI_F * rr_s, a_f + 4.0, float(LW - 1))
        pos = a_f[:, None] + (b_f - a_f)[:, None] * grid[None, :]   # (n, 32) 실수 index
        i0 = np.clip(np.floor(pos).astype(int), 0, LW - 2); w1 = pos - i0
        prv = np.r_[ii[0], ii[:-1]]                           # ★ **직전 박동**의 파형
        W = BEAT[prv][:, lq, :]
        seg = W[np.arange(len(ii))[:, None], i0] * (1 - w1) + \
              W[np.arange(len(ii))[:, None], i0 + 1] * w1
        tw = T[lq]
        tseg = tw[i0] * (1 - w1) + tw[i0 + 1] * w1            # ★ 템플릿도 같은 좌표로
        c = corr_to(seg, tseg)
        e = np.sqrt(((seg - seg.mean(-1, keepdims=True)) ** 2).mean(-1))
        te = np.sqrt(((tseg - tseg.mean(-1, keepdims=True)) ** 2).mean(-1)) + 1e-9
        res = np.abs(seg - tseg)
        amp = np.ptp(tseg, axis=-1) + 1e-9
        half = PT_NRS // 2
        el = np.sqrt(((seg[:, half:] - seg[:, half:].mean(-1, keepdims=True)) ** 2).mean(-1))
        tel = np.sqrt(((tseg[:, half:] - tseg[:, half:].mean(-1, keepdims=True)) ** 2)
                      .mean(-1)) + 1e-9
        col = np.c_[c, e / te, res.max(-1) / amp,
                    res.argmax(-1) / float(PT_NRS - 1), el / tel]
        # ★★★ 레코드 안에서 **RR 로 잔차화**(라벨 미사용 · R22).
        #   2차 다항으론 부족했다(스모크: 잔차 |ρ| 0.41) — 감사 지표가 **스피어만**이므로
        #   기저에 **RR 의 레코드 내 순위**(와 그 제곱)를 넣어 단조 성분까지 뗀다.
        z = (rr_s - rr_s.mean()) / (rr_s.std() + 1e-9)
        rk = rankdata(rr_s) / float(len(ii))
        A = np.c_[np.ones(len(ii)), z, z ** 2, z ** 3, rk, rk ** 2]
        coef, *_ = np.linalg.lstsq(A, col, rcond=None)
        out[ii] = col - A @ coef
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
T_FEAT = time.time()
MORPH = morph_feats(False); MORPHX = morph_feats(True); INTV = interval_feats()
VEC = vec_feats(); PONT2 = pont2_feats()
run.log(f"  ({time.time()-T_FEAT:.0f}초) 형태 8 + 구간 12 + **P 벡터축 6** + **RR정규화 P-on-T 5**열")
run.log(f"  ★★★ P 벡터축(2리드 · 임상 기준 ≥{AXIS_DEG:.0f}°) — " + " · ".join(VECNAMES))
run.log(f"  ★★★ RR 정규화 P-on-T(직전 R + {PT_LO_F}RR ~ min({PT_HI_F}RR, 창끝) · {PT_NRS}표본 · RR 잔차화) — " + " · ".join(PT2NAMES))

# ── ★★★ P0 자 검증 — 하나라도 깨지면 중단(R16)
POL = INTV[:, 9]; PABS = INTV[:, 8]; PF = INTV[:, 11]
is_v = np.isin(SYM, ("V", "E", "F")); is_n = np.isin(SYM, ("N", "L", "R", "e", "j", "n"))
pv = float(np.mean(PABS[is_v])) if is_v.any() else float("nan")
pn = float(np.mean(PABS[is_n])) if is_n.any() else float("nan")
inv_rate = float(np.mean(POL < 0))
fv = float(np.mean(PF[is_v])) if is_v.any() else float("nan")
fn = float(np.mean(PF[is_n])) if is_n.any() else float("nan")
THR_MED = float(np.median(list(P_THR.values())))
run.log(f"\n  ★★★ **P0 자 검증**(`sym` 은 **자 검증에만** 쓴다 — 적합·선택엔 안 쓴다)")
run.log(f"    QRS 폭 {QRS_MS:.1f}ms  범위 [{QRS_LO_MS}, {QRS_HI_MS}]  "
        + ("✅" if QRS_LO_MS <= QRS_MS <= QRS_HI_MS else "❌"))
run.log(f"    PR    {PR_MS:.1f}ms  범위 [{PR_LO_MS}, {PR_HI_MS}]  "
        + ("✅" if PR_LO_MS <= PR_MS <= PR_HI_MS else "❌"))
run.log(f"    **역위 P 비율 {inv_rate:.1%}** — 0 이면 `|corr|` 정렬이 작동 안 하는 것이다")
run.log(f"    **P 검출 타당성** |corr| — N 박동 {pn:.4f} vs **V 박동 {pv:.4f}** "
        + ("✅ V 가 낮다" if np.isfinite(pv) and pv < pn else "❌ V 가 안 낮다")
        + f"  (전체 검출률 {float(PF.mean()):.1%} · Q4-K {REF['q4k']['p_found']:.1%})")
run.log(f"    **검출률** — N 박동 {fn:.1%} vs **V 박동 {fv:.1%}** "
        + ("✅ V 가 낮다" if np.isfinite(fv) and fv < fn else "❌ V 가 안 낮다")
        + f"  · 레코드별 영점교정 문턱 중앙 **{THR_MED:.3f}**(표본치환 템플릿 {P_NULL_Q:.0%} 분위)")
_fail = []
if not (QRS_LO_MS <= QRS_MS <= QRS_HI_MS): _fail.append(f"QRS 폭 {QRS_MS:.1f}ms")
if not (PR_LO_MS <= PR_MS <= PR_HI_MS): _fail.append(f"PR {PR_MS:.1f}ms")
if not (PW_LO_MS <= PW_MS <= PW_HI_MS): _fail.append(f"P 폭 {PW_MS:.1f}ms(Q4-L 138.9)")
if not (inv_rate > 0.001): _fail.append("역위 P 0%")
if not (np.isfinite(pv) and np.isfinite(pn) and pv < pn):
    _fail.append(f"V |corr| {pv:.3f} >= N {pn:.3f}")
if not (np.isfinite(fv) and np.isfinite(fn) and fv < fn):
    _fail.append(f"V 검출률 {fv:.1%} >= N {fn:.1%} — 검출기가 P 유무를 반영 못 한다")
if _fail:
    raise AssetError("P0 자 검증 실패 — " + " · ".join(_fail)
                     + " ⇒ 고장난 자로 잰 결과는 무효다. 구간 축을 **접는다**(R16 · R35 ①)")

_rs = np.random.RandomState(SEED0 + 7)
def rec_shuffle(M):
    S_ = M.copy()
    for r in RS:
        ii = IDX_ALL[int(r)]; S_[ii] = M[ii][_rs.permutation(len(ii))]
    return S_
V_SH = rec_shuffle(VEC); T_SH = rec_shuffle(PONT2)
_moved = float(np.mean(np.any(np.abs(V_SH - VEC) > 1e-12, axis=1)))
if _moved < 0.5: raise AssetError(f"대조군이 거의 항등이다({_moved:.3f})(R35 ①)")
FEAT = {"base": F_BASE, "morph": np.c_[F_BASE, MORPH],
        "vec": np.c_[F_BASE, MORPH, VEC], "vshuf": np.c_[F_BASE, MORPH, V_SH],
        "pont2": np.c_[F_BASE, MORPH, PONT2], "p2shuf": np.c_[F_BASE, MORPH, T_SH],
        "comb": np.c_[F_BASE, MORPH, VEC, PONT2]}
for a, b in (("vec", "vshuf"), ("pont2", "p2shuf")):
    if FEAT[a].shape[1] != FEAT[b].shape[1]: raise AssetError(f"차원 대조군 불일치 {a}/{b}")
run.log("  차원 — " + " · ".join(f"{a} {FEAT[a].shape[1]}" for a in ARMS)
        + f" · 대조군 이동 {_moved:.1%}")

IDXS = {int(r): IDX_ALL[int(r)] for r in RS}
REC_OK = [int(r) for r in RS
          if TT_[IDXS[int(r)]].sum() >= MIN_S and (~TT_[IDXS[int(r)]]).sum() >= MIN_N]
BURD = {r: float(TT_[IDXS[r]].mean()) for r in REC_OK}
NS_ = {r: int(TT_[IDXS[r]].sum()) for r in REC_OK}
NRE = len(REC_OK)
run.log(f"  레코드 {len(RS)} · 채점 가능 **{NRE}**")

SLOPES, SLOPE_BY, _CUR = [], {}, [None]
def make_cal(s, y):
    lr = LogisticRegression(max_iter=3000, C=1e6).fit(np.asarray(s).reshape(-1, 1),
                                                       np.asarray(y).astype(int))
    a, b = float(lr.coef_[0, 0]), float(lr.intercept_[0])
    SLOPES.append(a)
    if _CUR[0] is not None: SLOPE_BY.setdefault(_CUR[0], []).append(a)
    return lambda v: a * np.asarray(v, float) + b

def split_rest(held):
    rest = sorted([r for r in REC_OK if r != held], key=lambda r: (BURD[r], r))
    dv = [r for i, r in enumerate(rest) if i % DEV_EVERY == 0]
    return [r for r in rest if r not in set(dv)], dv

def fit_fold(X, held, y_override):
    tr_r, dv_r = split_rest(held)
    tr = np.concatenate([IDXS[r] for r in tr_r]); dv = np.concatenate([IDXS[r] for r in dv_r])
    te = IDXS[held]
    Ftr = X[tr]; fmu = Ftr.mean(0); fsd = Ftr.std(0) + 1e-9
    ytr = TT_[tr].astype(int) if y_override is None else np.asarray(y_override[held], int)
    lr = LogisticRegression(max_iter=3000, C=1.0).fit((Ftr - fmu) / fsd, ytr)
    f = lambda ii: lr.decision_function((X[ii] - fmu) / fsd)
    raw = f(te)
    return te, make_cal(f(dv), TT_[dv])(raw), raw

def loro(X, y_override=None, tag=None):
    out = np.full(len(K), np.nan); raw = np.full(len(K), np.nan); _CUR[0] = tag
    for held in REC_OK:
        te, v, rv = fit_fold(X, held, y_override)
        out[te] = v; raw[te] = rv
    return out, raw

def per_auc(L):
    return {r: float(roc_auc_score(TT_[IDXS[r]].astype(int), L[IDXS[r]])) for r in REC_OK}
def per_ap(L):
    return {r: float(average_precision_score(TT_[IDXS[r]].astype(int), L[IDXS[r]]))
            for r in REC_OK}
def tp_at(L, r, k):
    idx = IDXS[r]; sc = L[idx]; yy = TT_[idx]
    k = int(min(max(1, k), len(idx)))
    fl = sc >= np.partition(sc, -k)[-k]
    return int((fl & yy).sum()), fl
# ★ 항등식 — 달성률 = TP / min(S, k). k <= S 면 **정밀도@k**, k > S 면 재현율
def ach_at(L, r, k):
    tp, _ = tp_at(L, r, k)
    return tp / max(1, min(NS_[r], int(k)))
def per_ksw(L):
    return {r: float(np.mean([ach_at(L, r, k) for k in K_SWEEP])) for r in REC_OK}
def per_ach(L):
    return {r: ach_at(L, r, MAIN_K) for r in REC_OK}
CONFIG["cohort"] = dict(n_ok=NRE, moved=_moved, qrs_ms=QRS_MS, pw_ms=PW_MS, pr_ms=PR_MS,
                        inv_rate=inv_rate, p_found=float(PF.mean()), p_corr_v=pv, p_corr_n=pn,
                        lead_split=LP_NE_LQ, p_found_v=fv, p_found_n=fn, p_thr=THR_MED,
                        q_on=Q_ON, p_pk=P_PK, p_on=P_ON, p_hi=P_HI, bite=BITE,
                        dims={a: int(FEAT[a].shape[1]) for a in ARMS})
run.save_json("config", CONFIG)


In [ ]:
# CELL 3 — 【R-A】 ★★★ R1 중복 감사 **v3**(자기 팔 전체에 대고) · 창 미끄러짐 계량 · 실행
run.log("\n" + "=" * 100)
run.log("【R-A】 ★★★ **R1 중복 감사 v3** — `base` 가 아니라 **`base+morph` 전체**에 대고 잰다")
run.log("=" * 100)
_ZB = (F_BASE - F_BASE.mean(0)) / (F_BASE.std(0) + 1e-9)
F_FULL = np.c_[F_BASE, MORPH]          # ★★★ 팔이 실제로 얹히는 바탕
run.log(f"  ★★★ Q4-M 은 `rank_dup()` 을 **`F_BASE`(RR {F_BASE.shape[1]}열)에만** 대고 쟀다. "
        f"정작 대비는 `morph + 새열` vs `morph + 셔플` 이었다 —")
run.log(f"     **`MORPH` 와의 중복은 한 번도 안 쟀다.** 이번엔 바탕을 "
        f"**{F_FULL.shape[1]}열(base+morph)** 로 넓힌다")

def _rank_dup_vs(col, M):
    """레코드 내 스피어만 |ρ| 의 **바탕 열 최댓값** — 단조 변환에 불변."""
    best = 0.0; arg = -1
    for j in range(M.shape[1]):
        # ★ 자기 자신은 제외한다 — `MORPH` 열을 `base+morph` 에 대고 재면 |ρ|=1 이 나온다
        if M.shape[0] == len(col) and np.allclose(M[:, j], col, rtol=0.0, atol=1e-12):
            continue
        v = []
        for r in REC_OK:
            ii = IDXS[r]
            if np.std(col[ii]) < 1e-12 or np.std(M[ii, j]) < 1e-12: continue
            rr = spearmanr(col[ii], M[ii, j]).statistic
            if np.isfinite(rr): v.append(abs(float(rr)))
        if v:
            m_ = float(np.median(v))
            if m_ > best: best, arg = m_, j
    return best, arg
FULLNAMES = [f"rr{j}" for j in range(F_BASE.shape[1])] + list(MNAMES)
def rank_dup(col):                      # ★ 하위호환 — base 만
    return _rank_dup_vs(col, F_BASE)[0]
def rank_dup_full(col):
    b, j = _rank_dup_vs(col, F_FULL)
    return b, (FULLNAMES[j] if 0 <= j < len(FULLNAMES) else "-")
def pooled_r2(col):
    y = np.asarray(col, float)
    if np.std(y) < 1e-12: return 1.0
    return float(max(0.0, min(1.0, LinearRegression().fit(_ZB, y).score(_ZB, y))))
def within_var(col):
    y = np.asarray(col, float); tot = np.std(y) + 1e-12
    return float(np.mean([np.std(y[IDXS[r]]) for r in REC_OK]) / tot)
def uni_one(col):
    a_ = []
    for r in REC_OK:
        ii = IDXS[r]; yy = TT_[ii].astype(int); v = col[ii]
        if 0 < yy.sum() < len(ii) and np.std(v) > 0:
            a_.append(abs(roc_auc_score(yy, v) - 0.5))
    return float(np.mean(a_)) if a_ else 0.0

run.log(f"\n  {'열':<15}{'블록':>6}{'|ρ|base':>9}{'|ρ|전체':>9}{'최근접':>16}"
        f"{'분산비':>7}{'단변량':>8}{'판정':>12}")
DUP = {}
for blk, M, names in (("형태", MORPH, MNAMES), ("구간", INTV, INAMES),
                      ("벡터", VEC, VECNAMES), ("PonT2", PONT2, PT2NAMES)):
    for j, nm in enumerate(names):
        rb = rank_dup(M[:, j]); rf, near = rank_dup_full(M[:, j])
        wv = within_var(M[:, j]); u = uni_one(M[:, j])
        if rf > DUP_RHO: vd = "⛔ 중복"
        elif wv < 0.15:  vd = "⚠️ 레코드상수"
        elif rf > DUP_FULL_WARN: vd = "⚠️ 근접"
        else: vd = "✅ 새 축"
        DUP[nm] = dict(block=blk, rho=rb, rho_full=rf, near=near, r2=pooled_r2(M[:, j]),
                       within=wv, uni=u, verdict=vd)
        run.log(f"  {nm:<15}{blk:>6}{rb:>9.4f}{rf:>9.4f}{near:>16}{wv:>7.3f}{u:>8.4f}{vd:>12}")
n_dup = sum(1 for v in DUP.values() if v["verdict"].startswith("⛔"))
n_near = sum(1 for v in DUP.values() if v["verdict"].startswith("⚠️ 근접"))
run.log(f"  ⇒ 중복 **{n_dup}** · 근접 **{n_near}** / {len(DUP)}열 "
        f"(문턱 ⛔ |ρ|전체 > {DUP_RHO} · ⚠️ > {DUP_FULL_WARN})")

# ── ★★★ 회고 ① — Q4-M 의 `prevTP_energy` 를 **원래 정의 그대로** 되만들어 v3 로 잰다
WTP_OLD = (min(LW - 12, W_T_S[1]), LW)
prevtp = np.zeros(len(K))
for r in RS:
    ii = IDX_ALL[int(r)]; B = BEAT[ii]; T = TMPL[int(r)]; lq = DELIN[int(r)]["lq"]
    sp = B[:, lq, WTP_OLD[0]:WTP_OLD[1]]; tsp = T[lq, WTP_OLD[0]:WTP_OLD[1]]
    ep = np.sqrt(((sp - sp.mean(-1, keepdims=True)) ** 2).mean(-1))
    tep = float(np.sqrt(((tsp - tsp.mean()) ** 2).mean())) + 1e-9
    v_ = ep / tep
    prevtp[ii] = np.r_[v_[0], v_[:-1]]
rb_o = rank_dup(prevtp); rf_o, near_o = rank_dup_full(prevtp); u_o = uni_one(prevtp)
run.log(f"\n  ★★★ **회고 ①** — Q4-M 의 `prevTP_energy` 를 v3 로 다시 잰다")
run.log(f"    |ρ| vs base **{rb_o:.4f}**(Q4-M {REF['q4m']['rho_base']['prevTP_energy']}) · "
        f"**|ρ| vs base+morph {rf_o:.4f}** (최근접 `{near_o}`) · 단변량 {u_o:.4f}"
        f"(Q4-M {REF['q4m']['uni']['prevTP_energy']})")
run.log(f"    ⇒ " + ("**바탕을 넓히니 중복이 드러난다 — Q4-M 관문이 자기와 무관한 것을 쟀다**"
                     if rf_o > rb_o + 0.05 else
                     "⚠️ 바탕을 넓혀도 |ρ| 가 안 오른다 — 원인 ① 진단은 이 열에선 약하다"))

# ── ★★★ 회고 ② — **창 미끄러짐을 `pre_rr` 로 직접 계량한다**(가설이 아니라 산술)
ms_ = lambda s_: (s_ - R_IDX) / FS * 1000.0
WTP_MS = (ms_(WTP_OLD[0]), ms_(WTP_OLD[1] - 1))     # 직전 R 기준 창(ms)
rr_ms_all = pre * FS / FS * 1000.0
p_on_ms = np.array([ms_(DELIN[int(r)]["p"][0]) for r in RS])
P_ON_MS = float(np.median(p_on_ms))                  # 현재 P 시작(현재 R 기준, 음수)
cross_p = float(np.mean(WTP_MS[1] - rr_ms_all > P_ON_MS))    # 창 끝이 현재 P 를 넘는가
cross_r = float(np.mean(WTP_MS[1] - rr_ms_all > 0.0))        # 창 끝이 현재 R 을 넘는가
run.log(f"\n  ★★★ **회고 ②** — Q4-M 창 미끄러짐 **계량**(산술 · `pre_rr` 실측)")
run.log(f"    박동 창 {ms_(0):+.1f} ~ {ms_(LW-1):+.1f}ms · `WTP` 표본 {WTP_OLD} = "
        f"**직전 R 기준 {WTP_MS[0]:+.1f} ~ {WTP_MS[1]:+.1f}ms**")
run.log(f"    RR 중앙 {float(np.median(rr_ms_all)):.1f}ms · 현재 P 시작 중앙 {P_ON_MS:+.1f}ms · "
        f"현재 q_on 중앙 {ms_(Q_ON):+.1f}ms")
run.log(f"    ▸ **창이 현재 박동의 P 를 넘어 들어간 박동 {cross_p:.1%}** · "
        f"**현재 R 까지 넘은 박동 {cross_r:.1%}**")
for _rr in (900, 800, 700, 600, 500):
    run.log(f"      RR {_rr}ms → 현재 R 기준 {WTP_MS[0]-_rr:+7.1f} ~ {WTP_MS[1]-_rr:+7.1f}ms")
run.log(f"    ⇒ " + ("**긴 RR 에선 T-P 기저선, 짧은 RR 에선 현재 P/QRS 를 읽는다 — "
                     "「RR 로 게이팅된 형태 재독」이 확인됐다**" if cross_p > 0.10 else
                     "⚠️ 미끄러짐이 작다 — 원인 ② 진단을 재검토해야 한다"))

# ── ★★ R3 1단계 — **RR 정규화가 실제로 누출을 뺐는가**(사전등록 문턱)
PT2_RHO = max(DUP[n]["rho"] for n in PT2NAMES)
PT2_RHOF = max(DUP[n]["rho_full"] for n in PT2NAMES)
PT2_FIXED = bool(PT2_RHO < PONT2_RHO_MAX)
run.log(f"\n  ★★ **R3 1단계(진단 검증)** — RR 정규화 P-on-T 의 |ρ| vs base 최대 "
        f"**{PT2_RHO:.4f}** (Q4-M `prevTP_energy` {REF['q4m']['rho_base']['prevTP_energy']} · "
        f"문턱 < {PONT2_RHO_MAX}) → " + ("✅ **누출이 빠졌다**" if PT2_FIXED else
                                         "❌ 누출이 안 빠졌다 — 2단계를 읽지 않는다(R29 ②)"))
run.log(f"     |ρ| vs base+morph 최대 {PT2_RHOF:.4f}")
VEC_RHO = max(DUP[n]["rho_full"] for n in VECNAMES)
run.log(f"  ★★★ **R2 입구** — P 벡터축의 |ρ| vs base+morph 최대 **{VEC_RHO:.4f}** "
        + ("✅ 새 축이다" if VEC_RHO <= DUP_FULL_WARN else "⚠️ 기존 열에 근접한다 — 증분을 그렇게 읽는다"))
g_("R1", "(입구 검사)",
   f"중복 {n_dup} · 근접 {n_near}/{len(DUP)} · 회고 `prevTP_energy` |ρ|base {rb_o:.4f} → "
   f"|ρ|전체 {rf_o:.4f}(`{near_o}`) · 창이 현재 P 를 넘은 박동 {cross_p:.1%} · "
   f"PonT2 |ρ|base {PT2_RHO:.4f} {'✅' if PT2_FIXED else '❌'} · 벡터 |ρ|전체 {VEC_RHO:.4f}")

run.log("\n" + "=" * 100)
run.log("【R-B】 실행")
run.log("=" * 100)
T0 = time.time()
L, LRAW = {}, {}
for a in ARMS:
    L[a], LRAW[a] = loro(FEAT[a], None, tag=a)
    run.log(f"  ({time.time()-T0:>5.0f}초) {a} 완료")
AUC = {a: per_auc(L[a]) for a in ARMS}
AP = {a: per_ap(L[a]) for a in ARMS}
KSW = {a: per_ksw(L[a]) for a in ARMS}
ACH = {a: per_ach(L[a]) for a in ARMS}
CAL_GAP = max(abs(AUC[a][r] - per_auc(LRAW[a])[r]) for a in ARMS for r in REC_OK)
bad_arms = []
for a in ARMS:
    sa = np.array(SLOPE_BY.get(a, []), float); mac = float(np.mean(list(AUC[a].values())))
    if len(sa) and mac > MIN_AUC_SLOPE and (np.median(sa) <= 0 or np.mean(sa <= 0) > MAX_NEG_SLOPE):
        bad_arms.append(a)
if bad_arms: raise AssetError(f"R0 실패 — 체계적 반전 {bad_arms}(R29 ②)")
g_("R0", "✅ 지지",
   f"QRS {QRS_MS:.1f}ms · PR {PR_MS:.1f}ms · P 폭 {PW_MS:.1f}ms · 역위 P {inv_rate:.1%} · "
   f"|corr| N {pn:.3f} > V {pv:.3f} · 검출률 N {fn:.1%} > V {fv:.1%} · 리드분리 {LP_NE_LQ:.1%}")

run.log(f"\n  {'팔':<9}{'차원':>5}{'k-스윕★':>10}{'문헌운영점':>12}{'달성률@300':>12}{'AUROC':>9}{'PR-AUC':>9}")
KLIT = {a: {r: float(np.mean([ach_at(L[a], r, k) for k in K_LIT])) for r in REC_OK}
        for a in ARMS}
for a in ARMS:
    run.log(f"  {a:<9}{FEAT[a].shape[1]:>5}{np.mean(list(KSW[a].values())):>10.4f}"
            f"{np.mean(list(KLIT[a].values())):>12.4f}"
            f"{np.mean(list(ACH[a].values())):>12.4f}{np.mean(list(AUC[a].values())):>9.4f}"
            f"{np.mean(list(AP[a].values())):>9.4f}")
run.log(f"  (Q4-M 앵커 k-스윕 — base {REF['q4m']['ksw']['base']} · morph "
        f"{REF['q4m']['ksw']['morph']} · comb {REF['q4m']['ksw']['comb']})")
run.log(f"  ▸ **운영점별 달성률**(base/morph/comb) — " + " · ".join(
    f"k={k} {np.mean([ach_at(L['base'], r, k) for r in REC_OK]):.3f}/"
    f"{np.mean([ach_at(L['morph'], r, k) for r in REC_OK]):.3f}/"
    f"{np.mean([ach_at(L['comb'], r, k) for r in REC_OK]):.3f}"
    for k in tuple(K_LIT) + tuple(K_SWEEP)))
run.log(f"  ▸ **형태 이득의 k 의존** — " + " · ".join(
    f"k={k} {np.mean([ach_at(L['morph'], r, k) - ach_at(L['base'], r, k) for r in REC_OK]):+.4f}"
    for k in tuple(K_LIT) + tuple(K_SWEEP))
    + f"  (Q4-M k=50 {REF['q4m']['k_gain']['k50']:+.4f} → k=300 {REF['q4m']['k_gain']['k300']:+.4f})")
CONFIG["R0"] = dict(qrs_ms=QRS_MS, pr_ms=PR_MS, pw_ms=PW_MS, inv_rate=inv_rate,
                    p_corr_v=pv, p_corr_n=pn, cal_gap=float(CAL_GAP), lead_split=LP_NE_LQ,
                    ksw={a: float(np.mean(list(KSW[a].values()))) for a in ARMS},
                    klit={a: float(np.mean(list(KLIT[a].values()))) for a in ARMS},
                    ach={a: float(np.mean(list(ACH[a].values()))) for a in ARMS},
                    auc={a: float(np.mean(list(AUC[a].values()))) for a in ARMS},
                    ap={a: float(np.mean(list(AP[a].values()))) for a in ARMS})
CONFIG["R1"] = dict(dup=DUP, retro_prevtp=dict(rho=rb_o, rho_full=rf_o, near=near_o, uni=u_o),
                    slide=dict(win_ms=list(WTP_MS), cross_p=cross_p, cross_r=cross_r,
                               rr_med=float(np.median(rr_ms_all)), p_on_ms=P_ON_MS),
                    pont2=dict(rho=PT2_RHO, rho_full=PT2_RHOF, fixed=PT2_FIXED),
                    vec_rho=VEC_RHO)
run.save_json("config", CONFIG)


In [ ]:
# CELL 4 — 【R-C】 영점 · ★★★ R2 주 관문(P 벡터축) · R3(RR 정규화 P-on-T)
run.log("\n" + "=" * 100)
run.log("【R-C】 영점(raw · rep 병기) · ★★★ **R2 — 2리드 P 벡터축이 형태 위에 얹히는가**")
run.log("=" * 100)
SRC = {"ksw": KSW, "auc": AUC, "ach": ACH}
NUL = {c[0]: {k: {r: [] for r in REC_OK} for k in ("ksw", "auc")} for c in CONTRASTS}
REPM = {c[0]: {"ksw": [], "auc": []} for c in CONTRASTS}
for s_ in range(N_PERM):
    rr = np.random.RandomState(SEED0 + 400 + s_)
    yov = {}
    for held in REC_OK:
        tr_r, _ = split_rest(held)
        tr = np.concatenate([IDXS[r] for r in tr_r])
        yov[held] = TT_[tr].astype(int)[rr.permutation(len(tr))]
    RW = {a: loro(FEAT[a], yov)[1] for a in ARMS}
    Sa = {"auc": {a: per_auc(RW[a]) for a in ARMS}, "ksw": {a: per_ksw(RW[a]) for a in ARMS}}
    for nm, x_, y_ in CONTRASTS:
        for key in ("ksw", "auc"):
            d = [Sa[key][y_][r] - Sa[key][x_][r] for r in REC_OK]
            REPM[nm][key].append(float(np.mean(d)))
            for i_, r in enumerate(REC_OK): NUL[nm][key][r].append(d[i_])
    run.log(f"    ({time.time()-T0:>5.0f}초) 영점 rep {s_+1}/{N_PERM}")
NSTAT, NREP = {}, {}
for nm, x_, y_ in CONTRASTS:
    NSTAT[nm], NREP[nm] = {}, {}
    for key in ("ksw", "auc"):
        NSTAT[nm][key] = boot_mean([float(np.mean(v)) for v in NUL[nm][key].values()],
                                   SEED0 + 61 + len(nm) + (0 if key == "ksw" else 7), NB_BOOT)
        v = np.array(REPM[nm][key], float)
        sd = float(v.std(ddof=1)) if len(v) > 1 else float("nan")
        se = sd / np.sqrt(max(1, len(v)))
        NREP[nm][key] = dict(mean=float(v.mean()), lo=float(v.mean() - 1.96 * se),
                             hi=float(v.mean() + 1.96 * se), n=int(len(v)))
NULL_BLUR = []
# ★ 보수적 문턱은 ✅ 를 어렵게 하는 건 맞지만 **❌ 를 만들어선 안 된다**(Q4-K 에서 넣은 규칙)
def two_verdicts(nm, key, obs):
    nhi = max(NSTAT[nm][key][2], NREP[nm][key]["hi"])
    thr = max(0.0, nhi) if np.isfinite(nhi) else float("nan")
    nh = max(mde(NSTAT[nm][key][1], NSTAT[nm][key][2]),
             mde(NREP[nm][key]["lo"], NREP[nm][key]["hi"]))
    blur = np.isfinite(nh) and np.isfinite(obs["mde"]) and nh > obs["mde"]
    def dem(v):
        if blur and v.startswith("❌"):
            NULL_BLUR.append((nm, key)); return "⚠️ 미결"
        return v
    return (dem(decide(obs["lo"], obs["hi"], thr, ">")), thr,
            dem(decide(obs["lo"], obs["hi"], nhi, ">")), nhi)

OBS, TAB = {}, {}
run.log(f"\n  {'대비':<15}{'Δ k-스윕 ★1차':>27}{'배포':>7}{'Δ AUROC(2차)':>25}{'배포':>7}")
for nm, x_, y_ in CONTRASTS:
    OBS[nm], TAB[nm] = {}, {}
    row = f"  {nm:<15}"
    for key in ("ksw", "auc"):
        m_, lo_, hi_, n_ = boot_pair([SRC[key][x_][r] for r in REC_OK],
                                     [SRC[key][y_][r] for r in REC_OK],
                                     SEED0 + 81 + len(nm) + (0 if key == "ksw" else 7), NB_BOOT)
        o = dict(mean=m_, lo=lo_, hi=hi_, n=int(n_), mde=float(mde(lo_, hi_)))
        OBS[nm][key] = o
        vd_, td_, vm_, tm_ = two_verdicts(nm, key, o)
        TAB[nm][key] = dict(obs=o, null_rec=list(NSTAT[nm][key][:3]), null_rep=NREP[nm][key],
                            thr_deploy=float(td_), thr_mech=float(tm_), v_deploy=vd_, v_mech=vm_)
        row += f"{m_:>+9.4f} [{lo_:+.4f},{hi_:+.4f}]{vd_:>7}"
    run.log(row)
if NULL_BLUR:
    run.log(f"  ⚠️ 영점 흐림으로 기각을 강등한 대비 {len(set(NULL_BLUR))}건 — "
            f"**증거 없음**이지 반대 증거가 아니다(R33 ①)")

om = OBS[MAIN_CT][PRIMARY]; vd, td, vm, tm = two_verdicts(MAIN_CT, PRIMARY, om)
oa = OBS[MAIN_CT][SECONDARY]; vd2, td2, _, _ = two_verdicts(MAIN_CT, SECONDARY, oa)
run.log(f"\n  ★★★ **R2 주 관문** — `{MAIN_CT}` 차원 {FEAT['vec'].shape[1]} 동일 "
        f"(2리드를 함께 쓰는 첫 특징)")
run.log(f"    **1차 k-스윕** Δ **{om['mean']:+.4f}** [{om['lo']:+.4f}, {om['hi']:+.4f}] · "
        f"영점 {NSTAT[MAIN_CT][PRIMARY][0]:+.4f} · 문턱 {td:+.4f} · MDE {om['mde']:.4f} → {vd}")
run.log(f"    2차 AUROC Δ {oa['mean']:+.4f} [{oa['lo']:+.4f}, {oa['hi']:+.4f}] → {vd2}")
run.log(f"    · `vec − morph` {OBS['vec-morph'][PRIMARY]['mean']:+.4f} "
        f"[{OBS['vec-morph'][PRIMARY]['lo']:+.4f}, {OBS['vec-morph'][PRIMARY]['hi']:+.4f}]")
g_("R2", vd,
   f"`vec` vs `vshuf` k-스윕 Δ {om['mean']:+.4f} [{om['lo']:+.4f}, {om['hi']:+.4f}] "
   f"(문턱 {td:+.4f}) · AUROC {oa['mean']:+.4f} {vd2} — "
   + ("**2리드 P/QRS 벡터축이 형태 위에 신호를 더한다**" if vd.startswith("✅") else
      ("벡터축도 형태 위에는 못 얹힌다 — 박동 단위 축이 포화다(R39 ①)" if vd.startswith("❌")
       else "가르지 못했다(R33 ①)")))

# ── ★★ R3 은 **2단 관문**이다 — 1단계(누출 제거)를 통과 못 하면 2단계를 안 읽는다(R29 ②)
v_pp = TAB["pont2-p2shuf"][PRIMARY]["v_deploy"]
v_cm = TAB["comb-morph"][PRIMARY]["v_deploy"]
if not PT2_FIXED:
    r3v = "❌ 기각"
    r3d = (f"1단계 실패 — RR 정규화 후에도 |ρ| vs base {PT2_RHO:.4f} >= {PONT2_RHO_MAX} "
           f"⇒ 2단계를 읽지 않는다. 원인 ② 진단이 틀렸거나 처방이 부족하다")
else:
    r3v = v_pp
    r3d = (f"1단계 ✅ |ρ| {PT2_RHO:.4f} < {PONT2_RHO_MAX}(Q4-M "
           f"{REF['q4m']['rho_base']['prevTP_energy']}) · 2단계 `pont2 − p2shuf` "
           f"{OBS['pont2-p2shuf'][PRIMARY]['mean']:+.4f} "
           f"[{OBS['pont2-p2shuf'][PRIMARY]['lo']:+.4f}, "
           f"{OBS['pont2-p2shuf'][PRIMARY]['hi']:+.4f}] {v_pp} · "
           f"`pont2 − morph` {OBS['pont2-morph'][PRIMARY]['mean']:+.4f}")
run.log(f"\n  ★★ **R3(2단 관문)** — {r3d}")
g_("R3", r3v, r3d)
run.log(f"  ★★ **`comb` vs `morph`(현재 최선 위에)** "
        f"{OBS['comb-morph'][PRIMARY]['mean']:+.4f} {v_cm} · 형태 재현 `morph − base` "
        f"{OBS['morph-base'][PRIMARY]['mean']:+.4f}(Q4-M {REF['q4m']['gates']['morph_base']:+.4f})")
CONFIG["R2"] = TAB
run.save_json("config", CONFIG)


In [ ]:
# CELL 5 — 【R-D】 열별 기여 · ★★★ R4 AAMI 범위 · R5 운영점/ESVEA · R6 딥러닝 3변형
run.log("\n" + "=" * 100)
run.log("【R-D】 열별 기여 · 위양성 구성")
run.log("=" * 100)
BLOCKS = {"P 벡터축": ("VEC", list(range(6))), "RR정규 P-on-T": ("PONT2", list(range(5)))}
run.log(f"  {'블록':<14}{'차원':>4}{'Δ k-스윕(형태 위에)':>28}{'Δ AUROC':>22}{'최고|ρ|전체':>11}")
R4B = {}
for nm, (src_, cols) in BLOCKS.items():
    M = VEC if src_ == "VEC" else PONT2
    names = VECNAMES if src_ == "VEC" else PT2NAMES
    Lb, _ = loro(np.c_[F_BASE, MORPH, M[:, cols]], None, tag=f"blk{src_}")
    kb = per_ksw(Lb); ub = per_auc(Lb)
    dk = boot_pair([KSW["morph"][r] for r in REC_OK], [kb[r] for r in REC_OK],
                   SEED0 + 200 + len(nm), NB_BOOT)
    du = boot_pair([AUC["morph"][r] for r in REC_OK], [ub[r] for r in REC_OK],
                   SEED0 + 230 + len(nm), NB_BOOT)
    rmax = max(DUP[names[j]]["rho_full"] for j in cols)
    R4B[nm] = dict(cols=[names[j] for j in cols], ksw=list(dk[:3]), auc=list(du[:3]),
                   rho_max=float(rmax))
    run.log(f"  {nm:<14}{len(cols):>4}{dk[0]:>+9.4f} [{dk[1]:+.4f},{dk[2]:+.4f}]"
            f"{du[0]:>+9.4f} [{du[1]:+.4f},{du[2]:+.4f}]{rmax:>11.3f}")
run.log(f"\n  ▸ **열별 단변량**(|AUROC−0.5|)")
for nm, names in (("벡터", VECNAMES), ("PonT2", PT2NAMES), ("형태", MNAMES)):
    run.log(f"    {nm:<8}" + " · ".join(f"{n} {DUP[n]['uni']:.4f}" for n in names))
run.log(f"    (Q4-M 앵커 — prevTP_energy {REF['q4m']['uni']['prevTP_energy']} · "
        f"clus_d_own {REF['q4m']['uni']['clus_d_own']} · "
        f"p_energy_ratio {REF['q4m']['uni']['p_energy_ratio']})")

# ══ ★★★ R4 — AAMI EC57 범위: AF 구간을 SVEB 평가에서 뺀다 ══════════════════════
run.log("\n" + "=" * 100)
run.log("【R-E】 ★★★ **R4 — AAMI EC57 범위**: AF 구간 무라벨 검출 → **타당성 관문**")
run.log("=" * 100)
run.log(f"  근거 — AAMI EC57 은 AF 에피소드를 SVEB 평가에서 **분자·분모 양쪽에서 제외**한다. "
        f"MIT-BIH 는 AF 안에서 A/a 를 안 매기고(ESC 2020: AF = P 파 완전 소실), 그래서 "
        f"Q4-J 의 「FN 이 AF 대리에 몰린다」는 **모델 실패가 아니라 채점 범위 오류**일 수 있다.")
run.log(f"  ⚠️ 리듬 라벨이 없다(Q7-D) ⇒ **무라벨 검출기**를 만들고 "
        f"**「검출 구간 안에서 S 밀도가 실제로 낮은가」** 를 타당성 관문으로 건다.")
AFB = np.zeros(len(K), bool)
AF_DIAG = {}
for r in RS:
    ii = IDX_ALL[int(r)]; n_ = len(ii)
    if n_ < AF_WIN: continue
    rr_ = pre[ii]; med_ = float(np.median(rr_)) + 1e-9
    pf_ = PF[ii]
    flag = np.zeros(n_, bool)
    for s0 in range(0, n_ - AF_WIN + 1, AF_STEP):
        w = slice(s0, s0 + AF_WIN)
        d_ = np.diff(rr_[w])
        rmssd = float(np.sqrt(np.mean(d_ ** 2))) / med_
        if rmssd > AF_RMSSD and float(np.mean(pf_[w])) < AF_PFOUND:
            flag[w] = True
    AFB[ii] = flag
    AF_DIAG[int(r)] = float(flag.mean())
af_frac = float(AFB.mean())
s_in = float(TT_[AFB].mean()) if AFB.any() else float("nan")
s_out = float(TT_[~AFB].mean()) if (~AFB).any() else float("nan")
run.log(f"\n  ▸ 검출 — AF 로 표시된 박동 **{af_frac:.1%}** · 레코드별 비율 중앙 "
        f"{float(np.median(list(AF_DIAG.values()))):.1%} · 최대 "
        f"{float(np.max(list(AF_DIAG.values()))):.1%}")
# ★★★ 타당성은 **레코드 안에서 짝지어** 잰다.
#   풀링 비교(안 vs 밖)는 **레코드 간 교란**을 탄다 — 검출이 저부담 레코드에 몰리면
#   실제 결핍이 0 이어도 「안이 낮다」로 보인다(널 스모크가 이걸 실제로 잡았다).
AF_IN, AF_OUT, AF_REC = [], [], []
for r in RS:
    ii = IDX_ALL[int(r)]; m = AFB[ii]
    if int(m.sum()) < 30 or int((~m).sum()) < 30: continue
    AF_REC.append(int(r)); AF_IN.append(float(TT_[ii][m].mean()))
    AF_OUT.append(float(TT_[ii][~m].mean()))
if len(AF_REC) >= 5:
    dmi, dlo, dhi, dn = boot_pair(AF_IN, AF_OUT, SEED0 + 911, NB_BOOT)   # 밖 − 안
else:
    dmi = dlo = dhi = float("nan"); dn = len(AF_REC)
run.log(f"  ▸ **타당성(레코드 내 짝지은 차 · 주 판정)** — 짝지을 수 있는 레코드 {dn} · "
        f"밖 − 안 = **{dmi:+.4f}** [{dlo:+.4f}, {dhi:+.4f}]")
run.log(f"    (참고 · 풀링 — 안 {s_in:.4f} vs 밖 {s_out:.4f}. ⚠️ 풀링은 **레코드 간 교란**을 "
        f"타므로 판정에 쓰지 않는다)")
AF_OK = bool(af_frac >= AF_MIN_FRAC and dn >= 5 and np.isfinite(dlo) and dlo > 0.0)
if af_frac < AF_MIN_FRAC:
    r5d = (f"검출기가 거의 아무것도 안 잡았다({af_frac:.1%} < {AF_MIN_FRAC:.0%}) — "
           f"**제외하지 않는다**. 문턱(RMSSD>{AF_RMSSD} · P검출<{AF_PFOUND})이 이 코호트엔 엄하다")
    r5v = "⚠️ 미결"
elif dn < 5:
    r5d = (f"레코드 안에서 안/밖을 모두 갖춘 레코드가 {dn}개뿐이라 짝지은 판정을 못 한다 — "
           f"**제외하지 않는다**")
    r5v = "⚠️ 미결"
elif not AF_OK:
    r5d = (f"레코드 내 짝지은 차 {dmi:+.4f} [{dlo:+.4f}, {dhi:+.4f}] 가 0 을 못 뗀다 ⇒ "
           f"검출 구간 안에서 S 가 **유의하게 줄지 않는다** — 이 구간은 AF 가 아니거나 "
           f"라벨 관행이 우리 가정과 다르다. **제외하지 않는다**")
    r5v = "❌ 기각" if (np.isfinite(dhi) and dhi < 0.0) else "⚠️ 미결"
else:
    r5d = (f"레코드 내 짝지은 차 **{dmi:+.4f}** [{dlo:+.4f}, {dhi:+.4f}] — 검출 구간 안에서 "
           f"S 가 유의하게 준다 ⇒ **AAMI 제외가 정당화된다**. 아래는 범위를 밝힌 병기 "
           f"수치다(성능 주장 아님)")
    r5v = "✅ 지지"
run.log(f"  ⇒ {r5v}  {r5d}")

KSW_A = {}
REC_A = []
if AF_OK:
    for r in REC_OK:
        ii = IDXS[r]; keep = ~AFB[ii]
        if int(TT_[ii][keep].sum()) >= MIN_S and int((~TT_[ii][keep]).sum()) >= MIN_N:
            REC_A.append(r)
    def ach_scoped(sc, r, k):
        ii = IDXS[r]; keep = ~AFB[ii]
        y = TT_[ii][keep]; v = sc[ii][keep]
        ns = int(y.sum())
        kk = int(min(max(1, k), len(v)))
        fl = v >= np.partition(v, -kk)[-kk]
        return int((fl & y).sum()) / max(1, min(ns, kk))
    run.log(f"\n  ▸ **AAMI 범위 재채점**(AF 박동을 분자·분모에서 제외) — 채점 레코드 "
            f"{len(REC_OK)} → **{len(REC_A)}**")
    run.log(f"    {'팔':<9}{'k-스윕(전체)':>14}{'k-스윕(AF제외)':>16}{'Δ':>10}")
    for a in ARMS:
        v0 = float(np.mean([KSW[a][r] for r in REC_A])) if REC_A else float("nan")
        v1 = (float(np.mean([np.mean([ach_scoped(L[a], r, k) for k in K_SWEEP])
                             for r in REC_A])) if REC_A else float("nan"))
        KSW_A[a] = dict(all=v0, scoped=v1, delta=v1 - v0)
        run.log(f"    {a:<9}{v0:>14.4f}{v1:>16.4f}{v1 - v0:>+10.4f}")
    run.log(f"    ⚠️ **이 Δ 는 성능 개선이 아니다** — 분모에서 빼면 수는 기계적으로 오른다. "
            f"의미는 **평가 범위가 AAMI 와 맞았다**는 것뿐이다(R38 ⑦)")
g_("R4", r5v, r5d + f" · AF 박동 {af_frac:.1%} · 짝지은 차 {dmi:+.4f} "
   f"[{dlo:+.4f}, {dhi:+.4f}] · 풀링 안 {s_in:.4f} / 밖 {s_out:.4f}")

# ══ ★★ R5 — 운영점(문헌) · ESVEA 환자 단위 ══════════════════════════════════════
run.log("\n" + "=" * 100)
run.log("【R-F】 ★★ **R5 — 문헌 운영점 · ESVEA 환자 단위 분류**")
run.log("=" * 100)
run.log(f"  ▸ 문헌 판독 운영점은 **환자당 상위 30~50 박동**이다(Top-50 에서 burden 추정 정확도 "
        f"≥95% 유지 · 판독시간 ≥80% 절감). 우리는 **k=300** 을 써 왔다.")
run.log(f"    {'팔':<9}" + "".join(f"{'k=' + str(k):>10}" for k in tuple(K_LIT) + tuple(K_SWEEP)))
for a in ("base", "morph", "vec", "pont2", "comb"):
    run.log(f"    {a:<9}" + "".join(
        f"{np.mean([ach_at(L[a], r, k) for r in REC_OK]):>10.4f}"
        for k in tuple(K_LIT) + tuple(K_SWEEP)))
run.log(f"    (달성률 = TP/min(S,k) — k ≤ S 면 **정밀도@k**, k > S 면 재현율)")

# ★ ESVEA — ≥30 PAC/시간. ⚠️ `MIN_S=25` 인 30분 레코드는 **이미 ≥50 PAC/h** 라
#   `REC_OK` 안에서는 표적이 퇴화한다. ⇒ **78 레코드 전부**에 대해 누출 없이 점수를 만든다:
#   레코드 r 을 채점할 때 학습은 **`REC_OK` 중 r 을 뺀 것**으로 한다(r 의 라벨은 안 쓴다).
def loro_all(X, tag=None):
    out = np.full(len(K), np.nan); _CUR[0] = tag
    for r in RS:
        r = int(r)
        tr_r = [q for q in REC_OK if q != r]
        tr = np.concatenate([IDXS[q] for q in tr_r])
        mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
        lr = LogisticRegression(max_iter=3000, C=1.0).fit((X[tr] - mu) / sd,
                                                          TT_[tr].astype(int))
        ii = IDX_ALL[r]
        out[ii] = lr.decision_function((X[ii] - mu) / sd)
    return out
DUR_H = {int(r): float(np.sum(pre[IDX_ALL[int(r)]]) / 3600.0) for r in RS}
NS_ALL = {int(r): int(TT_[IDX_ALL[int(r)]].sum()) for r in RS}
PACH = {int(r): NS_ALL[int(r)] / max(DUR_H[int(r)], 1e-6) for r in RS}
ESV = {int(r): int(PACH[int(r)] >= ESVEA_PER_HOUR) for r in RS}
RS_I = [int(r) for r in RS]
n_pos = int(sum(ESV.values()))
run.log(f"\n  ▸ **ESVEA**(≥{ESVEA_PER_HOUR:.0f} PAC/시간 → 신규 AF HR 2.78 · 뇌졸중 HR 2.40) — "
        f"레코드 길이 중앙 {float(np.median(list(DUR_H.values()))) * 60:.1f}분 · "
        f"PAC/시간 중앙 {float(np.median(list(PACH.values()))):.1f}")
run.log(f"    ⚠️ `MIN_S={MIN_S}` 인 30분 레코드는 이미 ≥{MIN_S / 0.5:.0f} PAC/h 라 "
        f"`REC_OK`({len(REC_OK)}명) 안에서는 표적이 퇴화한다 ⇒ **{len(RS_I)} 레코드 전부**로 "
        f"누출 없이(자기 라벨 미사용) 채점한다")
run.log(f"    ESVEA 양성 **{n_pos}/{len(RS_I)}** 명 "
        f"(`REC_OK` 안에서만 보면 {int(sum(ESV[r] for r in REC_OK))}/{len(REC_OK)})")
ESVR = {}
if 0 < n_pos < len(RS_I):
    yv = np.array([ESV[r] for r in RS_I])
    for a in ("base", "morph", "comb"):
        LA = loro_all(FEAT[a], tag=f"esv{a}")
        thr_g = float(np.median(LA[np.isfinite(LA)]))
        agg = {
            "top50_mean": [float(np.mean(np.sort(LA[IDX_ALL[r]])[-50:])) for r in RS_I],
            "frac_over": [float(np.mean(LA[IDX_ALL[r]] > thr_g)) for r in RS_I],
            "mean_prob": [float(np.mean(1.0 / (1.0 + np.exp(-np.clip(LA[IDX_ALL[r]], -30, 30)))))
                          for r in RS_I]}
        ESVR[a] = {k_: float(roc_auc_score(yv, np.array(v_))) for k_, v_ in agg.items()}
        run.log(f"    ({time.time()-T0:>5.0f}초) ESVEA `{a}` 완료")
    run.log(f"    {'팔':<9}{'top50평균':>12}{'문턱초과비율':>14}{'평균확률':>12}")
    for a, d in ESVR.items():
        run.log(f"    {a:<9}{d['top50_mean']:>12.4f}{d['frac_over']:>14.4f}"
                f"{d['mean_prob']:>12.4f}")
    run.log(f"    ▸ **박동 단위가 포화라도 환자 단위 표적은 아직 안 쟀다** — 이건 새 표적이다. "
            f"참고: Q4-G 의 burden 회귀 ρ 는 +0.1635 였다")
else:
    run.log(f"    ⏭️ ESVEA 한쪽 클래스가 비어 판정 불가(양성 {n_pos}/{len(RS_I)})")
g_("R5", "(측정)",
   f"문헌 운영점 k=30 에서 base {np.mean([ach_at(L['base'], r, 30) for r in REC_OK]):.4f} → "
   f"morph {np.mean([ach_at(L['morph'], r, 30) for r in REC_OK]):.4f} · "
   f"ESVEA 양성 {n_pos}/{len(RS_I)} · "
   + (f"환자 단위 AUROC 최고 {max(max(d.values()) for d in ESVR.values()):.4f}"
      if ESVR else "ESVEA 판정 불가"))

# ══ ★★ R6 — 딥러닝 3변형: 데드락 재현 / 초기화 수정 / 레코드 내 순위손실 ═════════
run.log("\n" + "=" * 100)
run.log("【R-G】 ★★ **R6 — 딥러닝 3변형**(관문 아님 · GPU 없으면 건너뛴다)")
run.log("=" * 100)
run.log(f"  ★★★ Q4-M 의 α=0 은 **결과가 아니라 초기화 데드락**이다 — "
        f"`logit = off + α·h(z)` 에서 ∂L/∂α = (∂L/∂logit)·h(z) 이고 "
        f"∂L/∂h_w = (∂L/∂logit)·α·z 인데 **둘 다 0 으로 초기화**했다")
run.log(f"     국소 확인 — h=0·α=0 → α {REF['dead']['zeros']['alpha']:+.4f} · "
        f"h=정상·α=0 → α **{REF['dead']['normal']['alpha']:+.4f}** "
        f"(AUROC {REF['dead']['normal']['auc0']} → {REF['dead']['normal']['auc1']})")
run.log(f"  ★★★ 그리고 **손실이 지표와 다르다** — 지표는 레코드 내 상위 k 인데 BCE 는 "
        f"**풀링 로그우도**다(Q4-E H2: 레코드별 상수 시프트는 레코드 내 지표를 0.0e+00 도 "
        f"못 바꾼다) ⇒ **레코드 내 pairwise 순위손실**을 붙인다")
DLR = {"ran": False, "reason": "", "res": {}, "alpha": {}, "gain": {}}
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as Fnn
    HAS_CUDA = torch.cuda.is_available()
except Exception as e:
    torch = None; HAS_CUDA = False; DLR["reason"] = f"torch 없음({e})"
if torch is None:
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}")
elif not HAS_CUDA and not SMOKE:
    DLR["reason"] = "CUDA 없음"
    run.log(f"  ⏭️ 건너뜀 — {DLR['reason']}. **런타임을 GPU 로 바꾸면 이 절이 돈다**")
else:
    dev = "cuda" if HAS_CUDA else "cpu"
    run.log(f"\n  ▶ 장치 **{dev}** · {DL_FOLDS}겹 · {DL_EPOCH}에폭 · wd {DL_WD} · "
            f"pos_weight 없음 · **α 초기값 0(하한 보장)**")
    ordr = sorted(REC_OK, key=lambda r: (BURD[r], r))
    FOLD = {r: i % DL_FOLDS for i, r in enumerate(ordr)}
    PREV = np.zeros(len(K), int); NEXT = np.zeros(len(K), int)
    for r in RS:
        ii = IDX_ALL[int(r)]
        PREV[ii] = np.r_[ii[0], ii[:-1]]; NEXT[ii] = np.r_[ii[1:], ii[-1]]
    RELP = np.clip(REL, 0.3, 2.0); RELN = np.clip(post / (BASE12 + 1e-9), 0.3, 2.0)
    def make_x(bi):
        w = np.concatenate([BEAT[PREV[bi]], BEAT[bi], BEAT[NEXT[bi]]], axis=1)
        rr = np.stack([np.repeat(RELP[bi][:, None], LW, 1),
                       np.repeat(RELN[bi][:, None], LW, 1)], axis=1)
        return np.concatenate([w, rr], axis=1).astype("float32")
    NCH = 3 * NL + 2
    class Boost(nn.Module):
        def __init__(self, init):
            super().__init__()
            self.c = nn.Sequential(
                nn.Conv1d(NCH, 24, 7, 2, 3), nn.GroupNorm(4, 24), nn.ReLU(),
                nn.Conv1d(24, 32, 5, 2, 2), nn.GroupNorm(4, 32), nn.ReLU(),
                nn.Conv1d(32, 32, 3, 2, 1), nn.GroupNorm(4, 32), nn.ReLU(),
                nn.AdaptiveAvgPool1d(1))
            self.e = nn.Linear(32, DL_EMB); self.h = nn.Linear(DL_EMB, 1)
            self.alpha = nn.Parameter(torch.zeros(1))     # ★ α=0 출발(하한 보장) — 항상
            if init == "zeros":                            # ★ Q4-M 재현 — 데드락
                nn.init.zeros_(self.h.weight); nn.init.zeros_(self.h.bias)
            else:                                          # ★★★ 고친 초기화
                nn.init.xavier_uniform_(self.h.weight); nn.init.zeros_(self.h.bias)
        def forward(self, x, off):
            z = torch.relu(self.e(self.c(x).squeeze(-1)))
            return off + self.alpha * self.h(z).squeeze(-1)
    def cpu_fold(X):
        sc = np.full(len(K), np.nan)
        for f in range(DL_FOLDS):
            te_r = [r for r in REC_OK if FOLD[r] == f]; tr_r = [r for r in REC_OK if FOLD[r] != f]
            tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
            mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-9
            lr = LogisticRegression(max_iter=3000, C=1.0).fit((X[tr] - mu) / sd,
                                                              TT_[tr].astype(int))
            sc[tr] = lr.decision_function((X[tr] - mu) / sd)
            sc[te] = lr.decision_function((X[te] - mu) / sd)
        return sc
    OFF = cpu_fold(FEAT["comb"])
    def run_variant(vname, init, loss_kind):
        sc_dl = np.full(len(K), np.nan); alphas = []
        for f in range(DL_FOLDS):
            te_r = [r for r in REC_OK if FOLD[r] == f]; tr_r = [r for r in REC_OK if FOLD[r] != f]
            tr = np.concatenate([IDXS[r] for r in tr_r]); te = np.concatenate([IDXS[r] for r in te_r])
            wmu = float(BEAT[tr].mean()); wsd = float(BEAT[tr].std()) + 1e-9
            torch.manual_seed(SEED0 + f)
            net = Boost(init).to(dev)
            opt = torch.optim.Adam(net.parameters(), 1e-3, weight_decay=DL_WD)
            bce = nn.BCEWithLogitsLoss()
            for ep in range(DL_EPOCH):
                net.train()
                rng = np.random.RandomState(SEED0 + f * 10 + ep)
                if loss_kind == "bce":
                    perm = rng.permutation(len(tr))
                    for b0 in range(0, len(tr), DL_BATCH):
                        bi = tr[perm[b0:b0 + DL_BATCH]]
                        x = torch.tensor((make_x(bi) - wmu) / wsd, device=dev)
                        o = torch.tensor(OFF[bi].astype("float32"), device=dev)
                        y = torch.tensor(TT_[bi].astype("float32"), device=dev)
                        opt.zero_grad(); bce(net(x, o), y).backward(); opt.step()
                else:
                    # ★★★ **레코드 내 pairwise 순위손실** — 지표(레코드 내 상위 k)와 같은 목적
                    for r in rng.permutation(np.array(tr_r)):
                        ii = IDXS[int(r)]
                        ip = ii[TT_[ii]]; inn = ii[~TT_[ii]]
                        if len(ip) < 2 or len(inn) < 2: continue
                        npos = int(min(DL_BATCH // 2, len(ip)))
                        nneg = int(min(DL_BATCH - npos, len(inn)))
                        bi = np.r_[rng.choice(ip, npos, replace=False),
                                   rng.choice(inn, nneg, replace=False)]
                        x = torch.tensor((make_x(bi) - wmu) / wsd, device=dev)
                        o = torch.tensor(OFF[bi].astype("float32"), device=dev)
                        s_ = net(x, o)
                        d = s_[:npos].unsqueeze(1) - s_[npos:].unsqueeze(0)
                        opt.zero_grad(); Fnn.softplus(-d).mean().backward(); opt.step()
            net.eval()
            with torch.no_grad():
                for b0 in range(0, len(te), 2048):
                    bi = te[b0:b0 + 2048]
                    x = torch.tensor((make_x(bi) - wmu) / wsd, device=dev)
                    o = torch.tensor(OFF[bi].astype("float32"), device=dev)
                    sc_dl[bi] = net(x, o).cpu().numpy()
            alphas.append(float(net.alpha.detach().cpu().numpy()[0]))
            run.log(f"    ({time.time()-T0:>5.0f}초) {vname} 겹 {f+1}/{DL_FOLDS} · "
                    f"α {alphas[-1]:+.4f}")
        return sc_dl, alphas
    base_ksw = [float(np.mean([ach_at(OFF, r, k) for k in K_SWEEP])) for r in REC_OK]
    DLR["res"]["cpu_comb"] = dict(
        ksw=float(np.mean(base_ksw)), auc=float(np.mean(list(per_auc(OFF).values()))))
    for vname, init, lk in DL_VARIANTS:
        sc_dl, alphas = run_variant(vname, init, lk)
        v_ksw = [float(np.mean([ach_at(sc_dl, r, k) for k in K_SWEEP])) for r in REC_OK]
        db = boot_pair(base_ksw, v_ksw, SEED0 + 301 + len(vname), NB_BOOT)
        DLR["res"][vname] = dict(ksw=float(np.mean(v_ksw)),
                                 auc=float(np.mean(list(per_auc(sc_dl).values()))),
                                 init=init, loss=lk)
        DLR["alpha"][vname] = [float(a) for a in alphas]
        DLR["gain"][vname] = list(db[:3])
    DLR["ran"] = True
    run.log(f"\n  {'변형':<14}{'초기화':>8}{'손실':>7}{'α 중앙':>10}{'k-스윕':>10}{'AUROC':>9}"
            f"{'Δ vs cpu':>22}")
    run.log(f"  {'cpu_comb':<14}{'-':>8}{'-':>7}{'-':>10}"
            f"{DLR['res']['cpu_comb']['ksw']:>10.4f}{DLR['res']['cpu_comb']['auc']:>9.4f}"
            f"{'(기준)':>22}")
    for vname, init, lk in DL_VARIANTS:
        d = DLR["res"][vname]; g = DLR["gain"][vname]
        run.log(f"  {vname:<14}{init:>8}{lk:>7}"
                f"{float(np.median(DLR['alpha'][vname])):>+10.4f}"
                f"{d['ksw']:>10.4f}{d['auc']:>9.4f}"
                f"{g[0]:>+9.4f} [{g[1]:+.4f},{g[2]:+.4f}]")
    a_dead = float(np.median(DLR["alpha"]["boost_dead"]))
    a_fix = float(np.median(DLR["alpha"]["boost_fix"]))
    DEAD_OK = bool(abs(a_dead) < 1e-6 and abs(a_fix) > 1e-6)
    run.log(f"\n  ★★★ **데드락 진단 검증** — `boost_dead` α {a_dead:+.6f} "
            f"(0 에 갇혀야 한다) vs `boost_fix` α {a_fix:+.6f} → "
            + ("✅ **초기화 데드락이 확정됐다 — Q4-M 의 「CNN 이 더할 게 없다」는 철회된다**"
               if DEAD_OK else
               "⚠️ 예측대로 안 갈렸다 — α=0 의 원인을 다시 봐야 한다"))
    run.log(f"  ▸ **α=0 출발이라 세 변형 모두 하한이 보장된다** — Δ<0 이면 **과적합**이지 "
            f"구조 문제가 아니다 (Q4-K 하이브리드 −0.0523 · Q4-M +0.0000)")
    DLR["dead_ok"] = DEAD_OK
g_("R6", "(관문 아님)",
   (" · ".join(f"{v} α {float(np.median(DLR['alpha'][v])):+.4f} Δ {DLR['gain'][v][0]:+.4f}"
               for v, _, _ in DL_VARIANTS)
    + f" · 데드락 확정 {DLR.get('dead_ok')}") if DLR["ran"] else f"건너뜀 ({DLR['reason']})")

VSET, NSET = ("V", "E", "F"), ("N", "L", "R", "e", "j", "n")
FPC = {}
for a in ("base", "morph", "vec", "pont2", "comb"):
    c = Counter()
    for r in REC_OK:
        idx = IDXS[r]; _, fl = tp_at(L[a], r, MAIN_K)
        for s_ in SYM[idx][fl & (~TT_[idx])]: c[str(s_)] += 1
    FPC[a] = dict(v=sum(c[s] for s in VSET), n=sum(c[s] for s in NSET), tot=sum(c.values()))
run.log(f"\n  ▸ **위양성 구성@300**  {'팔':<8}{'심실기원':>10}{'상심실정상':>12}{'전체':>9}")
for a, d in FPC.items():
    run.log(f"                       {a:<8}{d['v']:>10}{d['n']:>12}{d['tot']:>9}")
run.log(f"    (Q4-M 앵커 V — base {REF['q4m']['fp_v']['base']} · morph "
        f"{REF['q4m']['fp_v']['morph']} · comb {REF['q4m']['fp_v']['comb']})")
CONFIG["R4B"] = R4B
CONFIG["R4"] = dict(af_frac=af_frac, s_in=s_in, s_out=s_out, ok=AF_OK,
                    paired=dict(mean=dmi, lo=dlo, hi=dhi, n=dn),
                    per_rec=AF_DIAG, scoped=KSW_A, n_rec=len(REC_A))
CONFIG["R5"] = dict(klit={a: {str(k): float(np.mean([ach_at(L[a], r, k) for r in REC_OK]))
                              for k in tuple(K_LIT) + tuple(K_SWEEP)} for a in ARMS},
                    esvea=dict(n_pos=n_pos, n=len(RS_I), auroc=ESVR,
                               n_ok_pos=int(sum(ESV[r] for r in REC_OK)),
                               pac_h={str(r): PACH[r] for r in RS_I}))
CONFIG["R6"] = DLR; CONFIG["fp"] = FPC
run.save_json("config", CONFIG)


In [ ]:
# CELL 6 — 【R-H】 필요표본 · R7 검산표
run.log("\n" + "=" * 100)
run.log("【R-H】 필요표본 · 검산표")
run.log("=" * 100)
eff = om["mean"] - TAB[MAIN_CT][PRIMARY]["thr_deploy"]
n5 = need_super(NRE, om["mde"], eff); n8 = need_super(NRE, om["mde"], eff, True)
bad = (not np.isfinite(eff)) or abs(eff) < om["mde"]
run.log(f"  R2 주 관문(k-스윕) 효과-문턱 {eff:+.4f} · 반폭 {om['mde']:.4f} · n(50%) {n5:.0f} · "
        f"n(80%) {n8:.0f}  "
        + ("★ **해석 불가**(R41 ②)" if bad else
           ("이미 충분하다" if n8 <= NRE else "표본이 더 필요하다")))

_dlline = (" · ".join(f"{v} α {float(np.median(DLR['alpha'][v])):+.4f} "
                      f"Δ {DLR['gain'][v][0]:+.4f}" for v, _, _ in DL_VARIANTS)
           if DLR["ran"] else f"건너뜀({DLR['reason']})")
CHECK = [
    dict(claim="★★★ Q4-M 의 역설(단변량 최고 · 증분 0)의 원인 ① — **중복 감사가 자기 팔을 "
               "안 봤다**",
         num=f"Q4-M 의 `rank_dup()` 은 `F_BASE`({F_BASE.shape[1]}열)에만 대고 쟀는데 대비는 "
             f"`morph + 새열` vs `morph + 셔플` 이었다. v3 로 바탕을 {F_FULL.shape[1]}열로 "
             f"넓혀 `prevTP_energy` 를 회고 재판정: |ρ| base {rb_o:.4f} → **전체 {rf_o:.4f}** "
             f"(최근접 `{near_o}`) · 본 런 근접 판정 {n_near}열",
         assume=f"문턱 ⛔ {DUP_RHO} · ⚠️ {DUP_FULL_WARN} 를 **데이터 보기 전** 고정(R34 ②)",
         iffalse="★★ |ρ| 가 안 오르면 원인 ①은 이 열에선 약한 것이고, 그때는 원인 ②(창 "
                 "미끄러짐)가 단독 설명이 된다"),
    dict(claim="★★★ 원인 ② — **`prevTP_energy` 의 창이 RR 따라 현재 박동 위로 미끄러진다**"
               "(가설이 아니라 산술)",
         num=f"창 `{WTP_OLD}` = 직전 R 기준 {WTP_MS[0]:+.1f}~{WTP_MS[1]:+.1f}ms. 현재 R 기준 "
             f"위치는 RR 900ms 에서 {WTP_MS[0]-900:+.1f}~{WTP_MS[1]-900:+.1f}(T-P 기저선), "
             f"RR 700 에서 {WTP_MS[0]-700:+.1f}~{WTP_MS[1]-700:+.1f}(현재 P), "
             f"RR 500 에서 {WTP_MS[0]-500:+.1f}~{WTP_MS[1]-500:+.1f}(현재 QRS). "
             f"**실측: 현재 P 를 넘은 박동 {cross_p:.1%} · 현재 R 까지 넘은 박동 {cross_r:.1%}** "
             f"⇒ 이 열은 「RR 로 게이팅된 형태 재독」이고, RR 은 `base` 에 형태는 `morph` 에 "
             f"이미 있다 — 단변량 최고({REF['q4m']['uni']['prevTP_energy']})와 증분 0"
             f"({REF['q4m']['gates']['pont_pshuf']:+.4f})이 **같은 하나의 사실**이다",
         assume="박동 창은 R@index 100 · 360Hz · 길이 300(자산 고정) · `pre_rr` 는 실측",
         iffalse="★★ 미끄러짐이 작으면 진단이 틀린 것이다 — 그때 P-on-T 는 **진짜 기각**이다"),
    dict(claim=f"★★★ R2 주 관문(2리드 P 벡터축) — k-스윕 {om['mean']:+.4f} → {VERD['R2']}",
         num=f"두 팔 모두 {FEAT['vec'].shape[1]}차원 · **형태 위에** 얹었다 · 영점 "
             f"{NSTAT[MAIN_CT][PRIMARY][0]:+.4f}(rep {NREP[MAIN_CT][PRIMARY]['mean']:+.4f}) · "
             f"문턱 {td:+.4f} · AUROC 로는 {oa['mean']:+.4f} · |ρ|전체 최대 {VEC_RHO:.4f}",
         assume="벡터 성분은 **P 봉우리 시점의 등전위 대비 편차**(2리드) — 라벨을 안 쓴다(R22)",
         iffalse="★★ ❌ 면 **박동 단위 축이 포화**라는 뜻이고, 남는 건 **범위(AAMI)·운영점·"
                 "환자 단위 표적(ESVEA)** 이다"),
    dict(claim=f"★★ R3 2단 관문 — RR 정규화가 누출을 뺐는가 → 그러고도 얹히는가",
         num=f"1단계 |ρ| vs base 최대 **{PT2_RHO:.4f}** (Q4-M "
             f"{REF['q4m']['rho_base']['prevTP_energy']} · 문턱 <{PONT2_RHO_MAX}) "
             f"{'✅' if PT2_FIXED else '❌'} · 2단계 `pont2 − p2shuf` "
             f"{OBS['pont2-p2shuf'][PRIMARY]['mean']:+.4f} "
             f"[{OBS['pont2-p2shuf'][PRIMARY]['lo']:+.4f}, "
             f"{OBS['pont2-p2shuf'][PRIMARY]['hi']:+.4f}]",
         assume="창을 RR 의 분수(0.35~0.95RR)로 잡아 **현재 R 을 절대 안 넘고**, 32표본 "
                "리샘플로 모양만 보고, 레코드 안에서 RR 2차 다항으로 잔차화한다",
         iffalse="★★ 1단계를 못 넘으면 2단계를 **안 읽는다**(R29 ②) — 처방이 부족한 것이다"),
    dict(claim=f"★★★ R4 AAMI 범위 — AF 제외의 **타당성** {VERD['R4']}",
         num=f"AF 로 표시된 박동 {af_frac:.1%} · **레코드 내 짝지은 차(밖−안) "
             f"{dmi:+.4f} [{dlo:+.4f}, {dhi:+.4f}] · 짝 {dn}명** · 채점 레코드 "
             f"{len(REC_OK)} → {len(REC_A)}. 풀링은 안 {s_in:.4f} / 밖 {s_out:.4f} 이지만 "
             f"**레코드 간 교란**을 타므로 판정에 안 쓴다(널 스모크가 풀링만 쓰면 "
             f"거짓 통과함을 실제로 보였다)",
         assume=f"무라벨 검출기(창 {AF_WIN}박 · RMSSD/중앙 > {AF_RMSSD} **그리고** P 검출률 "
                f"< {AF_PFOUND})를 데이터 보기 전 고정. MIT-BIH 는 AF 안에서 A/a 를 안 매긴다",
         iffalse="★★★ **이건 성능 관문이 아니다** — 분모에서 빼면 수는 기계적으로 오른다. "
                 "S 밀도가 안 낮으면 **제외하지 않는다**(R38 ⑦)"),
    dict(claim="★★ R5 — **운영점이 한 자릿수 틀렸다** · ESVEA 는 새 표적이다",
         num=f"문헌 운영점은 환자당 상위 30~50 인데 우리는 k=300 을 썼다. k=30 에서 base "
             f"{np.mean([ach_at(L['base'], r, 30) for r in REC_OK]):.4f} → morph "
             f"{np.mean([ach_at(L['morph'], r, 30) for r in REC_OK]):.4f} · "
             f"k=300 에서 {np.mean(list(ACH['base'].values())):.4f} → "
             f"{np.mean(list(ACH['morph'].values())):.4f}. ESVEA 양성 {n_pos}/{len(RS_I)}"
             + (f" · 환자 단위 AUROC 최고 "
                f"{max(max(d.values()) for d in ESVR.values()):.4f}" if ESVR else ""),
         assume=f"ESVEA = ≥{ESVEA_PER_HOUR:.0f} PAC/시간(Binici 2010 — 신규 AF HR 2.78 · "
                f"뇌졸중 HR 2.40) · 레코드 길이는 `pre_rr` 합 · **78 레코드 전부**로 채점",
         iffalse="★ 박동 단위가 포화라도 **환자 단위 표적은 아직 안 쟀다** — 거기서 여지가 "
                 "남아 있으면 그쪽이 다음 축이다"),
    dict(claim="★★★ R6 — 딥러닝의 α=0 은 **결과가 아니라 초기화 데드락**이다",
         num=f"`logit = off + α·h(z)` 에서 ∂L/∂α = (∂L/∂logit)·h(z) · ∂L/∂h_w = "
             f"(∂L/∂logit)·α·z 인데 Q4-M 은 **둘 다 0** 으로 초기화했다. 국소 확인 "
             f"h=0·α=0 → α {REF['dead']['zeros']['alpha']:+.4f}(AUROC "
             f"{REF['dead']['zeros']['auc0']}→{REF['dead']['zeros']['auc1']}) vs "
             f"h=정상·α=0 → α {REF['dead']['normal']['alpha']:+.4f}(AUROC "
             f"{REF['dead']['normal']['auc0']}→{REF['dead']['normal']['auc1']}). "
             f"본 런 3변형 — {_dlline}",
         assume="세 변형 모두 **α=0 출발**이라 초기 상태가 정확히 `cpu_comb` 다 — 하한 보장",
         iffalse="★★★ `boost_dead` 가 α=0 을 재현하고 `boost_fix` 가 안 그러면 진단 확정이고 "
                 "**Q4-M 의 P5 해석은 철회**된다. 그래도 Δ<0 이면 그건 **과적합**이다"),
    dict(claim="★★ 손실 함수가 지표와 어긋나 있었다 — **레코드 내 순위손실**",
         num="지표는 레코드 내 상위 k 인데 BCE 는 풀링 로그우도다. Q4-E H2 가 이미 증명했다 — "
             "레코드별 **상수 시프트**는 레코드 내 지표를 **0.0e+00** 만큼도 못 바꾼다. "
             "즉 BCE 가 쓰는 용량의 일부는 지표가 **정의상 무시하는 방향**이다. "
             f"`boost_rank` 는 레코드 안에서 (양성, 음성) 쌍의 softplus 마진을 최적화한다"
             + (f" — Δ {DLR['gain']['boost_rank'][0]:+.4f}" if DLR["ran"] else ""),
         assume="쌍은 **같은 레코드 안에서만** 만든다 — 레코드 간 눈금은 손실에 안 들어간다",
         iffalse="★ `boost_rank` 가 `boost_fix` 를 못 이기면 목적함수 어긋남은 이 규모에선 "
                 "지배적 요인이 아니다"),
    dict(claim="★★ 형태는 여전히 **유일하게 확립된 축**이다",
         num=f"`morph − base` k-스윕 {OBS['morph-base'][PRIMARY]['mean']:+.4f}"
             f"(Q4-L·Q4-M {REF['q4m']['gates']['morph_base']:+.4f} — **네 번째 재현**) · "
             f"위양성 심실기원 {FPC['base']['v']} → {FPC['morph']['v']} · "
             f"이득의 k 의존 k=30 "
             f"{np.mean([ach_at(L['morph'], r, 30) - ach_at(L['base'], r, 30) for r in REC_OK]):+.4f}"
             f" → k=300 "
             f"{np.mean(list(ACH['morph'].values())) - np.mean(list(ACH['base'].values())):+.4f}",
         assume="같은 코호트·같은 LORO — 다섯 런의 수치가 직접 비교 가능하다",
         iffalse="★ 벡터축도 안 얹히면 **박동 단위 특징은 포화**이고, 남는 건 범위·운영점·"
                 "환자 단위 표적이다"),
]
for i, ck in enumerate(CHECK, 1):
    run.log(f"\n  [{i}] **{ck['claim']}**")
    run.log(f"      근거   {ck['num']}")
    run.log(f"      가정   {ck['assume']}")
    run.log(f"      틀리면 {ck['iffalse']}")
CONFIG["need"] = dict(effect=float(eff), half=float(om["mde"]), sup50=float(n5),
                      sup80=float(n8), uninterpretable=bool(bad))
CONFIG["R7"] = CHECK
run.save_json("config", CONFIG)


In [ ]:
# CELL 7 — 【R-I】 그림 · 요약 · 마무리
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
fig, ax = plt.subplots(1, 3, figsize=(16.5, 4.6))

KS_ALL = tuple(K_LIT) + tuple(K_SWEEP)
for a, st in (("base", "o--"), ("morph", "s-"), ("comb", "^-")):
    ax[0].plot(KS_ALL, [np.mean([ach_at(L[a], r, k) for r in REC_OK]) for k in KS_ALL],
               st, label=a)
ax[0].axvspan(30, 50, color="tab:orange", alpha=.15)
ax[0].text(38, 0.02, "literature\noperating point", fontsize=7, ha="center",
           transform=ax[0].get_xaxis_transform())
ax[0].set_xscale("log"); ax[0].set_xticks(KS_ALL)
ax[0].set_xticklabels([str(k) for k in KS_ALL], fontsize=7)
ax[0].set_xlabel("k (beats reviewed per patient)"); ax[0].set_ylabel("achievement")
ax[0].legend(fontsize=7); ax[0].grid(alpha=.3)
ax[0].set_title("R5 : operating point", fontsize=9)

nm = [c[0] for c in CONTRASTS]
vv = [OBS[n][PRIMARY]["mean"] for n in nm]
lo = [vv[i] - OBS[n][PRIMARY]["lo"] for i, n in enumerate(nm)]
hi = [OBS[n][PRIMARY]["hi"] - vv[i] for i, n in enumerate(nm)]
ax[1].errorbar(vv, np.arange(len(nm)), xerr=[lo, hi], fmt="o", capsize=5, color="tab:blue")
ax[1].scatter([NSTAT[n][PRIMARY][0] for n in nm], np.arange(len(nm)), marker="x", s=45,
              color="tab:gray", label="null (raw)")
ax[1].axvline(0, color="k", lw=.9)
ax[1].set_yticks(range(len(nm))); ax[1].set_yticklabels(nm, fontsize=7)
ax[1].set_xlabel("k-sweep achievement delta"); ax[1].legend(fontsize=7)
ax[1].set_title("R2/R3 : contrasts", fontsize=9); ax[1].grid(alpha=.3, axis="x")

if DLR["ran"]:
    vn = ["cpu_comb"] + [v for v, _, _ in DL_VARIANTS]
    ax[2].bar(np.arange(len(vn)), [DLR["res"][v]["ksw"] for v in vn],
              color=["tab:gray", "tab:red", "tab:green", "tab:blue"][:len(vn)])
    for i, v in enumerate(vn):
        if v in DLR["alpha"]:
            ax[2].text(i, DLR["res"][v]["ksw"],
                       f"α={float(np.median(DLR['alpha'][v])):+.2f}",
                       ha="center", va="bottom", fontsize=7)
    ax[2].set_xticks(range(len(vn))); ax[2].set_xticklabels(vn, fontsize=7, rotation=20)
    ax[2].set_ylabel("k-sweep"); ax[2].grid(alpha=.3, axis="y")
    ax[2].set_title("R6 : deadlock / init fix / rank loss", fontsize=9)
else:
    xs = np.arange(len(ARMS))
    ax[2].bar(xs, [np.mean(list(KSW[a].values())) for a in ARMS], color="tab:blue")
    ax[2].set_xticks(xs); ax[2].set_xticklabels(ARMS, fontsize=7, rotation=20)
    ax[2].set_title("arms (deep learning skipped)", fontsize=9); ax[2].grid(alpha=.3, axis="y")
fig.tight_layout()
PNG = run.save_fig("q4n_scope_rank_vector", fig)
plt.close(fig); display(Image(PNG))

run.log("\n" + "=" * 100)
run.log("요약")
run.log("=" * 100)
ok_ = lambda k: VERD.get(k, "").startswith("✅")
for g in READ_ORDER[:6]:
    run.log(f"  {g:<5}{VERD.get(g, '(관문 아님)')}")
run.log("")
run.log(f"  ★★★ **R2 주 관문(2리드 P 벡터축)** — `{MAIN_CT}` k-스윕 {om['mean']:+.4f} "
        f"[{om['lo']:+.4f}, {om['hi']:+.4f}] → {VERD['R2']}")
run.log(f"  ★★★ **R1 원인 규명** — `prevTP_energy` |ρ| base {rb_o:.4f} → **전체 {rf_o:.4f}** "
        f"(`{near_o}`) · **창이 현재 P 를 넘은 박동 {cross_p:.1%} · 현재 R 까지 {cross_r:.1%}**")
run.log(f"  ★★ **R3** — RR 정규화 |ρ| {PT2_RHO:.4f} "
        f"{'✅ 누출 제거' if PT2_FIXED else '❌ 누출 잔존'} · `pont2 − p2shuf` "
        f"{OBS['pont2-p2shuf'][PRIMARY]['mean']:+.4f} → {VERD['R3']}")
run.log(f"  ★★★ **R4 AAMI 범위** — AF 박동 {af_frac:.1%} · 레코드 내 짝지은 차 "
        f"{dmi:+.4f} [{dlo:+.4f}, {dhi:+.4f}] → {VERD['R4']}")
run.log(f"  ★★ **R5 운영점** — k=30 base "
        f"{np.mean([ach_at(L['base'], r, 30) for r in REC_OK]):.4f} → morph "
        f"{np.mean([ach_at(L['morph'], r, 30) for r in REC_OK]):.4f} · ESVEA 양성 "
        f"{n_pos}/{len(RS_I)}"
        + (f" · 환자 단위 AUROC 최고 {max(max(d.values()) for d in ESVR.values()):.4f}"
           if ESVR else ""))
run.log(f"  ★★★ **R6 딥러닝** — " + (
    " · ".join(f"{v} α {float(np.median(DLR['alpha'][v])):+.4f} Δ {DLR['gain'][v][0]:+.4f}"
               for v, _, _ in DL_VARIANTS)
    + f" · **데드락 확정 {DLR.get('dead_ok')}**" if DLR["ran"] else f"건너뜀({DLR['reason']})"))
run.log(f"  ▸ k-스윕 — " + " · ".join(f"{a} {np.mean(list(KSW[a].values())):.4f}" for a in ARMS))
run.log(f"  ▸ 위양성 심실기원 — " + " · ".join(f"{a} {FPC[a]['v']}" for a in FPC))
run.log(f"  ▸ 형태 재현 — `morph − base` {OBS['morph-base'][PRIMARY]['mean']:+.4f} "
        f"(Q4-M {REF['q4m']['gates']['morph_base']:+.4f})")

run.finish({
    "exp_id": "quest46_q4n_scope_rank_vector",
    "metric": "ksweep_vec_minus_vshuf",
    "value": float(om["mean"]),
    "passed": bool(ok_("R0") and ok_("R2")),
    "summary": ("Q4-M 의 역설(단변량 최고인 열의 증분이 0)의 원인을 특정하고 고쳤다 — "
                "① 중복 감사가 `F_BASE` 에만 대고 쟀는데 팔은 `morph` 위에 얹었고 "
                "② `prevTP_energy` 의 창이 RR 따라 현재 박동의 P·QRS 위로 미끄러져 "
                "「RR 로 게이팅된 형태 재독」이 됐다(둘 다 이미 팔에 있는 정보다). "
                "③ 군집은 팔 안의 특징 공간에서 군집했다. 그래서 중복 감사를 "
                "**`base+morph` 전체**에 대고 다시 만들고, P-on-T 를 **RR 정규화**로 "
                "다시 시험하며, 새 축으로 **2리드 P/QRS 벡터축**(임상 기준 축이동 ≥30° · "
                "지금까지 모든 P 특징이 한 리드만 봤다)을 넣었다. "
                "그리고 딥러닝의 α=0 은 결과가 아니라 **초기화 데드락**이었다 — α 와 h 를 "
                "둘 다 0 으로 초기화해 서로의 기울기를 0 에 가뒀다. 초기화를 고치고, "
                "**레코드 내 pairwise 순위손실**로 목적함수를 지표에 맞춘 3변형으로 판정한다. "
                "문헌을 받아 **AAMI EC57 범위(AF 구간 제외)** 와 **문헌 운영점(환자당 상위 "
                "30~50 · 우리는 300 이었다)**, **ESVEA 환자 단위 표적**을 처음 넣었다."),
    "verdicts": VERD, "notes": NOTE, "rule_check": RULE_CHECK,
    "cohort": CONFIG.get("cohort", {}), "R0": CONFIG.get("R0", {}),
    "R1": CONFIG.get("R1", {}), "R2": CONFIG.get("R2", {}), "R4B": CONFIG.get("R4B", {}),
    "R4": CONFIG.get("R4", {}), "R5": CONFIG.get("R5", {}), "R6": CONFIG.get("R6", {}),
    "fp": CONFIG.get("fp", {}), "need": CONFIG.get("need", {}),
    "R7": CONFIG.get("R7", []), "fig": PNG})
run.log(f"\n저장 완료 — {run.dir}")
run.log("다음: `python pipelines/ingest_run.py --results result.json "
        "--notebook notebooks/quest46_q4n_scope_rank_vector.ipynb`")
